# Feature Engineering

The goal of this phase is to transform raw data into a set of meaningful 
features that capture the predictive signal hidden across the 7 tables.

The process is structured in 5 blocks:

1. **Cleaning & fixing** — correct known anomalies before creating any feature
2. **Main table features** — ratios, temporal features, EXT_SOURCE combinations
3. **Secondary table aggregations** — aggregate bureau, previous applications, installments, POS, credit card per client
4. **Cross-table features** — combine signals from multiple tables
5. **Categorical encoding** — target encoding for high-cardinality columns

All transformations are implemented in DuckDB SQL to avoid loading large 
secondary tables into pandas memory.

## Imports

In [1]:
import duckdb
import pandas as pd
import numpy as np

# 1. Connect to DuckDB (file-based for persistence)
con = duckdb.connect(database='data/credit_risk.duckdb')

# 2. Define the path and file mapping for all datasets
# Using a dictionary for better maintainability
data_path = 'data/raw/'

datasets = {
    'app_train': 'application_train.csv',
    'app_test': 'application_test.csv',
    'bureau': 'bureau.csv',
    'bureau_balance': 'bureau_balance.csv',
    'prev_app': 'previous_application.csv',
    'pos_cash': 'POS_CASH_balance.csv',
    'installments': 'installments_payments.csv',
    'credit_card': 'credit_card_balance.csv'
}

# 3. Register all files as DuckDB Views
print("Registering datasets as SQL Views...")
for view_name, file_name in datasets.items():
    full_path = f"{data_path}{file_name}"
    try:
        con.execute(f"CREATE OR REPLACE VIEW {view_name} AS SELECT * FROM read_csv_auto('{full_path}')")
        print(f"  [OK] View '{view_name}' registered.")
    except Exception as e:
        print(f"  [ERROR] Could not register {view_name}: {e}")

# 4. Quick sanity check: count rows for the main tables
print("\n--- Summary ---")
summary_query = """
    SELECT 'app_train'    AS table_name, COUNT(*) AS row_count FROM app_train
    UNION ALL
    SELECT 'app_test'     AS table_name, COUNT(*) AS row_count FROM app_test
    UNION ALL
    SELECT 'bureau'       AS table_name, COUNT(*) AS row_count FROM bureau
    UNION ALL
    SELECT 'bureau_balance' AS table_name, COUNT(*) AS row_count FROM bureau_balance
    UNION ALL
    SELECT 'prev_app'     AS table_name, COUNT(*) AS row_count FROM prev_app
    UNION ALL
    SELECT 'pos_cash'     AS table_name, COUNT(*) AS row_count FROM pos_cash
    UNION ALL
    SELECT 'installments' AS table_name, COUNT(*) AS row_count FROM installments
    UNION ALL
    SELECT 'credit_card'  AS table_name, COUNT(*) AS row_count FROM credit_card
"""
display(con.execute(summary_query).df())

Registering datasets as SQL Views...
  [OK] View 'app_train' registered.
  [OK] View 'app_test' registered.
  [OK] View 'bureau' registered.
  [OK] View 'bureau_balance' registered.
  [OK] View 'prev_app' registered.
  [OK] View 'pos_cash' registered.
  [OK] View 'installments' registered.
  [OK] View 'credit_card' registered.

--- Summary ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,table_name,row_count
0,app_train,307511
1,app_test,48744
2,bureau,1716428
3,bureau_balance,27299925
4,prev_app,1670214
5,pos_cash,10001358
6,installments,13605401
7,credit_card,3840312


In [19]:
# All the tables persistent in file-based DuckDB, ready for analysis
con.execute("SHOW TABLES").fetchdf()

# With rows number, for a quick check
con.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_type = 'BASE TABLE'
    ORDER BY table_name
""").fetchdf()

,table_name
0,app_test_final
1,app_train_features_tbl
2,app_train_final
3,app_train_final_v2
4,app_train_final_v3
5,app_train_temp1
6,cc_features
7,installments_features


## Block 1 — Cleaning & Fixing

Before creating any feature, known data quality issues in `app_train` are corrected.
The output is a new table `app_train_clean` used by all subsequent blocks.

**Operations applied:**

- `DAYS_EMPLOYED = 365243` → set to `NULL` (Home Credit's sentinel for "not employed")  
  and create binary flag `IS_NOT_EMPLOYED`
- `DAYS_*` columns → converted to positive values (absolute days ago)
- `AMT_*` columns → capped at p99.9% computed on training data only  
  (prevents outliers from distorting ratios downstream)
- `OWN_CAR_AGE` → forced to `NULL` when `FLAG_OWN_CAR = 'N'`  
  (a car age for a client with no car is meaningless)
- Categorical anomalies `XNA`, `XAP`, `Unknown` → replaced with `NULL`

**Why this matters:**  
Any feature built on uncleaned values inherits the anomaly.  
`DAYS_EMPLOYED = 365243` would produce a `CREDIT_TERM` of thousands of months —  
a nonsensical value that the model would try to learn from.

In [3]:
# All cleaning is done via DuckDB SQL — no pandas loading into memory.
# Output is a new view 'app_train_clean' used by all subsequent FE blocks.

def build_clean_view(con):
    """
    Block 1 — Cleaning & fixing before feature engineering.
    Creates app_train_clean view with:
      1. DAYS_EMPLOYED anomaly fixed  → NULL + IS_NOT_EMPLOYED flag
      2. AMT_INCOME_TOTAL capped      → at p99.9%
      3. DAYS_* standardized          → converted to positive (abs value)
      4. Anomalous categoricals       → XNA / XAP → NULL
      5. OWN_CAR_AGE                  → NULL when no car
    """

    # ── 1. Compute caps from training data only ───────────────────────
    # Must be computed before building the view to avoid leakage
    caps = con.execute("""
        SELECT
            PERCENTILE_CONT(0.999) WITHIN GROUP
                (ORDER BY AMT_INCOME_TOTAL)    AS income_cap,
            PERCENTILE_CONT(0.999) WITHIN GROUP
                (ORDER BY AMT_CREDIT)          AS credit_cap,
            PERCENTILE_CONT(0.999) WITHIN GROUP
                (ORDER BY AMT_ANNUITY)         AS annuity_cap,
            PERCENTILE_CONT(0.999) WITHIN GROUP
                (ORDER BY AMT_GOODS_PRICE)     AS goods_cap
        FROM app_train
    """).fetchdf().iloc[0]

    income_cap  = caps['income_cap']
    credit_cap  = caps['credit_cap']
    annuity_cap = caps['annuity_cap']
    goods_cap   = caps['goods_cap']

    print("Caps computed (p99.9%):")
    print(f"  AMT_INCOME_TOTAL  : {income_cap:>15,.0f}")
    print(f"  AMT_CREDIT        : {credit_cap:>15,.0f}")
    print(f"  AMT_ANNUITY       : {annuity_cap:>15,.0f}")
    print(f"  AMT_GOODS_PRICE   : {goods_cap:>15,.0f}")

    # ── 2. Categorical values to recode as NULL ───────────────────────
    # XNA / XAP are Home Credit's codes for "not applicable" / "not available"
    # Keeping them as strings would create spurious categories
    CAT_COLS_TO_CLEAN = [
        "NAME_TYPE_SUITE",
        "OCCUPATION_TYPE",
        "FONDKAPREMONT_MODE",
        "HOUSETYPE_MODE",
        "WALLSMATERIAL_MODE",
        "EMERGENCYSTATE_MODE",
    ]

    xna_exprs = []
    for col in CAT_COLS_TO_CLEAN:
        xna_exprs.append(f"""
            CASE WHEN {col} IN ('XNA', 'XAP', 'Unknown')
                 THEN NULL ELSE {col}
            END AS {col}
        """)

    # ── 3. Build the cleaned view ─────────────────────────────────────
    con.execute(f"""
        CREATE OR REPLACE VIEW app_train_clean AS
        SELECT
            -- ── Keys & target ────────────────────────────────────────
            SK_ID_CURR,
            TARGET,

            -- ── DAYS_EMPLOYED: fix encoding anomaly ──────────────────
            -- 365243 is HC's sentinel for "not employed / pensioner"
            CASE WHEN DAYS_EMPLOYED = 365243 THEN NULL
                 ELSE ABS(DAYS_EMPLOYED)
            END                                         AS DAYS_EMPLOYED,
            CASE WHEN DAYS_EMPLOYED = 365243 THEN 1
                 ELSE 0
            END                                         AS IS_NOT_EMPLOYED,

            -- ── DAYS_* → positive values (days ago) ──────────────────
            ABS(DAYS_BIRTH)                             AS DAYS_BIRTH,
            ABS(DAYS_REGISTRATION)                      AS DAYS_REGISTRATION,
            ABS(DAYS_ID_PUBLISH)                        AS DAYS_ID_PUBLISH,
            ABS(DAYS_LAST_PHONE_CHANGE)                 AS DAYS_LAST_PHONE_CHANGE,

            -- ── AMT_* → capped at p99.9% ─────────────────────────────
            LEAST(AMT_INCOME_TOTAL,  {income_cap})      AS AMT_INCOME_TOTAL,
            LEAST(AMT_CREDIT,        {credit_cap})      AS AMT_CREDIT,
            LEAST(AMT_ANNUITY,       {annuity_cap})     AS AMT_ANNUITY,
            LEAST(AMT_GOODS_PRICE,   {goods_cap})       AS AMT_GOODS_PRICE,

            -- ── OWN_CAR_AGE → NULL when client has no car ────────────
            CASE WHEN FLAG_OWN_CAR = 'N' THEN NULL
                 ELSE OWN_CAR_AGE
            END                                         AS OWN_CAR_AGE,

            -- ── Categorical anomalies → NULL ─────────────────────────
            {', '.join(xna_exprs)},

            -- ── Structural missing → HAS_APPRAISAL flag ──────────────────
            -- These ~40 columns miss together when no property appraisal
            -- was performed. The missingness itself is informative.
            CASE WHEN APARTMENTS_AVG IS NULL THEN 0 ELSE 1 END  AS HAS_APPRAISAL,

            -- Pass through the appraisal columns raw (imputed at model time)
            APARTMENTS_AVG,
            APARTMENTS_MEDI,
            APARTMENTS_MODE,
            BASEMENTAREA_AVG,
            BASEMENTAREA_MEDI,
            BASEMENTAREA_MODE,
            YEARS_BEGINEXPLUATATION_AVG,
            YEARS_BEGINEXPLUATATION_MEDI,
            YEARS_BEGINEXPLUATATION_MODE,
            YEARS_BUILD_AVG,
            YEARS_BUILD_MEDI,
            YEARS_BUILD_MODE,
            COMMONAREA_AVG,
            COMMONAREA_MEDI,
            COMMONAREA_MODE,
            ELEVATORS_AVG,
            ELEVATORS_MEDI,
            ELEVATORS_MODE,
            ENTRANCES_AVG,
            ENTRANCES_MEDI,
            ENTRANCES_MODE,
            FLOORSMAX_AVG,
            FLOORSMAX_MEDI,
            FLOORSMAX_MODE,
            FLOORSMIN_AVG,
            FLOORSMIN_MEDI,
            FLOORSMIN_MODE,
            LANDAREA_AVG,
            LANDAREA_MEDI,
            LANDAREA_MODE,
            LIVINGAPARTMENTS_AVG,
            LIVINGAPARTMENTS_MEDI,
            LIVINGAPARTMENTS_MODE,
            LIVINGAREA_AVG,
            LIVINGAREA_MEDI,
            LIVINGAREA_MODE,
            NONLIVINGAPARTMENTS_AVG,
            NONLIVINGAPARTMENTS_MEDI,
            NONLIVINGAPARTMENTS_MODE,
            NONLIVINGAREA_AVG,
            NONLIVINGAREA_MEDI,
            NONLIVINGAREA_MODE,
            TOTALAREA_MODE,

            -- ── All remaining columns passed through unchanged ────────
            -- (binary flags, EXT_SOURCE, region cols, etc.)
            CODE_GENDER,
            NAME_CONTRACT_TYPE,
            NAME_INCOME_TYPE,
            NAME_EDUCATION_TYPE,
            NAME_FAMILY_STATUS,
            NAME_HOUSING_TYPE,
            FLAG_OWN_CAR,
            FLAG_OWN_REALTY,
            CNT_CHILDREN,
            CNT_FAM_MEMBERS,
            EXT_SOURCE_1,
            EXT_SOURCE_2,
            EXT_SOURCE_3,
            REGION_RATING_CLIENT,
            REGION_RATING_CLIENT_W_CITY,
            REGION_POPULATION_RELATIVE,
            REG_CITY_NOT_WORK_CITY,
            REG_CITY_NOT_LIVE_CITY,
            REG_REGION_NOT_WORK_REGION,
            LIVE_CITY_NOT_WORK_CITY,
            LIVE_REGION_NOT_WORK_REGION,
            FLAG_EMP_PHONE,
            FLAG_WORK_PHONE,
            FLAG_CONT_MOBILE,
            FLAG_PHONE,
            FLAG_EMAIL,
            FLAG_MOBIL,
            HOUR_APPR_PROCESS_START,
            DEF_30_CNT_SOCIAL_CIRCLE,
            DEF_60_CNT_SOCIAL_CIRCLE,
            OBS_30_CNT_SOCIAL_CIRCLE,
            OBS_60_CNT_SOCIAL_CIRCLE,
            OCCUPATION_TYPE,
            ORGANIZATION_TYPE,
            AMT_REQ_CREDIT_BUREAU_HOUR,
            AMT_REQ_CREDIT_BUREAU_DAY,
            AMT_REQ_CREDIT_BUREAU_WEEK,
            AMT_REQ_CREDIT_BUREAU_MON,
            AMT_REQ_CREDIT_BUREAU_QRT,
            AMT_REQ_CREDIT_BUREAU_YEAR,

            -- FLAG_DOCUMENT_* (all 20)
            FLAG_DOCUMENT_2,  FLAG_DOCUMENT_3,  FLAG_DOCUMENT_4,
            FLAG_DOCUMENT_5,  FLAG_DOCUMENT_6,  FLAG_DOCUMENT_7,
            FLAG_DOCUMENT_8,  FLAG_DOCUMENT_9,  FLAG_DOCUMENT_10,
            FLAG_DOCUMENT_11, FLAG_DOCUMENT_12, FLAG_DOCUMENT_13,
            FLAG_DOCUMENT_14, FLAG_DOCUMENT_15, FLAG_DOCUMENT_16,
            FLAG_DOCUMENT_17, FLAG_DOCUMENT_18, FLAG_DOCUMENT_19,
            FLAG_DOCUMENT_20, FLAG_DOCUMENT_21

        FROM app_train
    """)

    print("\nView 'app_train_clean' created.")

build_clean_view(con)

Caps computed (p99.9%):
  AMT_INCOME_TOTAL  :         900,000
  AMT_CREDIT        :       2,517,300
  AMT_ANNUITY       :         110,048
  AMT_GOODS_PRICE   :       2,250,000

View 'app_train_clean' created.


In [4]:
# ── Sanity checks ─────────────────────────────────────────────────────

def sanity_check_clean_view(con):
    """
    Verifies the cleaning was applied correctly:
      - DAYS_EMPLOYED anomaly is gone
      - AMT_INCOME_TOTAL is capped
      - IS_NOT_EMPLOYED flag counts match original anomaly count
      - No more XNA/XAP in categorical columns
    """
    checks = {
        "DAYS_EMPLOYED anomaly removed":
            "SELECT COUNT(*) FROM app_train_clean WHERE DAYS_EMPLOYED > 100000",

        "IS_NOT_EMPLOYED flag count":
            "SELECT SUM(IS_NOT_EMPLOYED) FROM app_train_clean",

        "Original anomaly count (should match above)":
            "SELECT COUNT(*) FROM app_train WHERE DAYS_EMPLOYED = 365243",

        "AMT_INCOME_TOTAL max after cap":
            "SELECT MAX(AMT_INCOME_TOTAL) FROM app_train_clean",

        "AMT_INCOME_TOTAL max before cap":
            "SELECT MAX(AMT_INCOME_TOTAL) FROM app_train",

        "XNA in OCCUPATION_TYPE after clean":
            "SELECT COUNT(*) FROM app_train_clean WHERE OCCUPATION_TYPE = 'XNA'",

        "OWN_CAR_AGE nulls for non-car owners":
            """SELECT COUNT(*) FROM app_train_clean
               WHERE FLAG_OWN_CAR = 'N' AND OWN_CAR_AGE IS NOT NULL""",
        
        "HAS_APPRAISAL distribution (0/1)":
            """SELECT CONCAT(
                SUM(1 - HAS_APPRAISAL), ' no appraisal / ',
                SUM(HAS_APPRAISAL), ' with appraisal'
            ) FROM app_train_clean""",
    }

    print("Sanity checks on app_train_clean\n")
    print(f"{'Check':<50} {'Result':>12}")
    print("─" * 65)
    for label, query in checks.items():
        result = con.execute(query).fetchone()[0]
        status = ""
        if "anomaly removed" in label and result > 0:
            status = "  ⚠ FAIL"
        elif "XNA" in label and result > 0:
            status = "  ⚠ FAIL"
        elif "OWN_CAR_AGE nulls" in label and result > 0:
            status = "  ⚠ FAIL"
        else:
            status = "  ✓"
        try:
            print(f"{label:<50} {result:>12,}{status}")
        except (ValueError, TypeError):
            print(f"{label:<50} {str(result):>30}{status}")

sanity_check_clean_view(con)

Sanity checks on app_train_clean

Check                                                    Result
─────────────────────────────────────────────────────────────────
DAYS_EMPLOYED anomaly removed                                 0  ✓
IS_NOT_EMPLOYED flag count                               55,374  ✓
Original anomaly count (should match above)              55,374  ✓
AMT_INCOME_TOTAL max after cap                        900,000.0  ✓
AMT_INCOME_TOTAL max before cap                    117,000,000.0  ✓
XNA in OCCUPATION_TYPE after clean                            0  ✓
OWN_CAR_AGE nulls for non-car owners                          0  ✓
HAS_APPRAISAL distribution (0/1)                   156061 no appraisal / 151450 with appraisal  ✓


## Block 2 — Feature Engineering from Main Table

Creates `app_train_features` view by deriving new features from `app_train_clean`.  
No secondary tables are involved here — all features come from the 122 original columns.

**Five groups of features created:**

**1. Financial ratios**  
Relationships between loan amount, annuity, goods price and income.  
Tree-based models cannot compute ratios by themselves — these must be 
explicit features. Key ones: `CREDIT_INCOME_RATIO`, `ANNUITY_INCOME_RATIO`, `CREDIT_TERM`.

**2. Temporal features**  
`DAYS_*` columns converted to years and combined to produce derived signals  
such as `EMPLOYMENT_START_AGE` (age when the client started working) and  
`EMPLOYED_TO_LIFE_RATIO` (fraction of life spent employed).

**3. EXT_SOURCE combinations**  
`EXT_SOURCE_1/2/3` are external credit scores — the strongest predictors in the dataset.  
Their mean, product, min, max and std are computed to capture both the  
average signal and the disagreement between sources.  
`NULLIF` protects all divisions; `COALESCE` handles missing sources gracefully.

**4. Flag aggregations**  
`TOTAL_DOCS_PROVIDED` sums all 20 `FLAG_DOCUMENT_*` columns into a single count.  
`ANY_SOCIAL_CIRCLE_DEF` flags clients whose social circle has had defaults.  
`ENQUIRIES_SHORT_TERM` captures recent credit bureau pressure (last hour/day/week).

**5. Categorical encoding**  
- Binary columns (`FLAG_OWN_CAR`, `CODE_GENDER`) → direct 0/1 encoding  
- Ordinal columns (`NAME_EDUCATION_TYPE`) → integer scale preserving order  
- Low-cardinality nominals (`NAME_CONTRACT_TYPE`, `NAME_HOUSING_TYPE`) → one-hot  
- High-cardinality columns (`OCCUPATION_TYPE`, `ORGANIZATION_TYPE`) → passed raw,  
  to be target-encoded inside cross-validation at modelling time to avoid leakage

In [18]:
def build_main_table_features(con):
    """
    Block 2 — Feature engineering from app_train_clean.
    Creates view 'app_train_features' with all derived features:
      1. Financial ratios
      2. Temporal features from DAYS_*
      3. EXT_SOURCE combinations
      4. Aggregated flags
      5. Categorical encoding
    All operations are pure SQL — no pandas, no memory issues.
    """

    con.execute("""
        CREATE OR REPLACE VIEW app_train_features AS
        SELECT
            SK_ID_CURR,
            TARGET,

            -- ── 1. Financial ratios ───────────────────────────────────
            -- How much of income goes to repayment
            ROUND(AMT_ANNUITY  / NULLIF(AMT_INCOME_TOTAL, 0), 4)
                                                    AS ANNUITY_INCOME_RATIO,
            -- Total loan vs income
            ROUND(AMT_CREDIT   / NULLIF(AMT_INCOME_TOTAL, 0), 4)
                                                    AS CREDIT_INCOME_RATIO,
            -- Implicit loan duration in months
            ROUND(AMT_CREDIT   / NULLIF(AMT_ANNUITY, 0), 2)
                                                    AS CREDIT_TERM,
            -- What fraction of the loan covers the actual good
            ROUND(AMT_GOODS_PRICE / NULLIF(AMT_CREDIT, 0), 4)
                                                    AS GOODS_CREDIT_RATIO,
            -- Gap between loan and goods price (fees / insurance)
            AMT_CREDIT - AMT_GOODS_PRICE            AS CREDIT_GOODS_DIFF,
            -- Income per family member
            ROUND(AMT_INCOME_TOTAL / NULLIF(CNT_FAM_MEMBERS, 0), 2)
                                                    AS INCOME_PER_PERSON,
            -- Annuity per family member
            ROUND(AMT_ANNUITY / NULLIF(CNT_FAM_MEMBERS, 0), 2)
                                                    AS ANNUITY_PER_PERSON,
            -- Annuity relative to goods price
            ROUND(AMT_ANNUITY / NULLIF(AMT_GOODS_PRICE, 0), 4)  AS ANNUITY_GOODS_RATIO,

            -- ── 2. Temporal features ──────────────────────────────────
            ROUND(DAYS_BIRTH       / 365.25, 2)     AS AGE_YEARS,
            ROUND(DAYS_REGISTRATION / 365.25, 2)    AS REGISTRATION_YEARS,
            ROUND(DAYS_ID_PUBLISH   / 365.25, 2)    AS ID_PUBLISH_YEARS,
            ROUND(DAYS_LAST_PHONE_CHANGE / 365.25, 2) AS PHONE_CHANGE_YEARS,

            -- Employed years (NULL for non-employed)
            ROUND(DAYS_EMPLOYED / 365.25, 2)        AS EMPLOYED_YEARS,
            IS_NOT_EMPLOYED,
            
            -- ID publish years relative to age (proxy for residential instability)
            ROUND(
                (DAYS_ID_PUBLISH / 365.25) / NULLIF(DAYS_BIRTH / 365.25, 0), 4
            )                                           AS ID_PUBLISH_TO_AGE,
                
            -- Age when started working
            ROUND(
                (DAYS_BIRTH - DAYS_EMPLOYED) / 365.25, 2
            )                                       AS EMPLOYMENT_START_AGE,

            -- What % of life has been spent employed
            ROUND(
                DAYS_EMPLOYED / NULLIF(DAYS_BIRTH, 0), 4
            )                                       AS EMPLOYED_TO_LIFE_RATIO,

            -- How recently was ID published relative to registration
            DAYS_ID_PUBLISH - DAYS_REGISTRATION     AS ID_REGISTRATION_GAP,

            -- ── 3. EXT_SOURCE combinations ───────────────────────────
            -- Mean — robust when one source is missing
            ROUND(
                (COALESCE(EXT_SOURCE_1, 0) +
                 COALESCE(EXT_SOURCE_2, 0) +
                 COALESCE(EXT_SOURCE_3, 0))
                / NULLIF(
                    (CASE WHEN EXT_SOURCE_1 IS NOT NULL THEN 1 ELSE 0 END +
                     CASE WHEN EXT_SOURCE_2 IS NOT NULL THEN 1 ELSE 0 END +
                     CASE WHEN EXT_SOURCE_3 IS NOT NULL THEN 1 ELSE 0 END), 0
                ), 4
            )                                       AS EXT_SOURCE_MEAN,

            -- Product — penalises low scores on any source
            ROUND(
                COALESCE(EXT_SOURCE_1, 1) *
                COALESCE(EXT_SOURCE_2, 1) *
                COALESCE(EXT_SOURCE_3, 1), 6
            )                                       AS EXT_SOURCE_PROD,

            -- Min / Max — worst and best score across sources
            LEAST(
                COALESCE(EXT_SOURCE_1, 1),
                COALESCE(EXT_SOURCE_2, 1),
                COALESCE(EXT_SOURCE_3, 1)
            )                                       AS EXT_SOURCE_MIN,

            GREATEST(
                COALESCE(EXT_SOURCE_1, 0),
                COALESCE(EXT_SOURCE_2, 0),
                COALESCE(EXT_SOURCE_3, 0)
            )                                       AS EXT_SOURCE_MAX,

            -- Disagreement between sources (high std = inconsistent signals)
            ROUND(
                SQRT((
                    POWER(COALESCE(EXT_SOURCE_1,0) - (
                        COALESCE(EXT_SOURCE_1,0)+COALESCE(EXT_SOURCE_2,0)+COALESCE(EXT_SOURCE_3,0)
                    ) / 3.0, 2) +
                    POWER(COALESCE(EXT_SOURCE_2,0) - (
                        COALESCE(EXT_SOURCE_1,0)+COALESCE(EXT_SOURCE_2,0)+COALESCE(EXT_SOURCE_3,0)
                    ) / 3.0, 2) +
                    POWER(COALESCE(EXT_SOURCE_3,0) - (
                        COALESCE(EXT_SOURCE_1,0)+COALESCE(EXT_SOURCE_2,0)+COALESCE(EXT_SOURCE_3,0)
                    ) / 3.0, 2)
                ) / 3.0), 4
            )                                       AS EXT_SOURCE_STD,

            -- Pairwise diffs — disagreement between specific source pairs
            ROUND(EXT_SOURCE_1 - EXT_SOURCE_2, 4)  AS EXT_12_DIFF,
            ROUND(EXT_SOURCE_2 - EXT_SOURCE_3, 4)  AS EXT_23_DIFF,

            -- Interaction with financial variables
            ROUND(EXT_SOURCE_2 * AMT_INCOME_TOTAL, 2) AS EXT2_INCOME,
            ROUND(EXT_SOURCE_2 / NULLIF(CREDIT_INCOME_RATIO, 0), 4)
                                                    AS EXT2_CREDIT_RATIO,
            
            -- EXT_SOURCE: count of missing sources per client
            (CASE WHEN EXT_SOURCE_1 IS NULL THEN 1 ELSE 0 END +
            CASE WHEN EXT_SOURCE_2 IS NULL THEN 1 ELSE 0 END +
            CASE WHEN EXT_SOURCE_3 IS NULL THEN 1 ELSE 0 END)  AS EXT_MISSING_COUNT,

            -- EXT_SOURCE_3 × age interaction
            ROUND(EXT_SOURCE_3 * (DAYS_BIRTH / 365.25), 4)      AS EXT3_AGE,

            -- Has car but no real estate (asset inconsistency)
            CASE WHEN FLAG_OWN_CAR = 'Y'
                AND FLAG_OWN_REALTY = 'N'
                THEN 1 ELSE 0 END                               AS HAS_CAR_NOT_REALTY,

            -- Annuity exceeds 35% of income (financial stress threshold)
            CASE WHEN AMT_ANNUITY / NULLIF(AMT_INCOME_TOTAL, 0) > 0.35
                THEN 1 ELSE 0 END                               AS HIGH_ANNUITY_BURDEN,

            -- Loan amount exceeds goods price (client is taking extra cash)
            CASE WHEN AMT_CREDIT > AMT_GOODS_PRICE
                THEN 1 ELSE 0 END                               AS CREDIT_EXCEEDS_GOODS,

            -- Pass through raw EXT_SOURCE for the model too
            EXT_SOURCE_1,
            EXT_SOURCE_2,
            EXT_SOURCE_3,

            -- ── 4. Flag aggregations ──────────────────────────────────
            -- Total documents provided
            (COALESCE(FLAG_DOCUMENT_2,  0) + COALESCE(FLAG_DOCUMENT_3,  0) +
             COALESCE(FLAG_DOCUMENT_4,  0) + COALESCE(FLAG_DOCUMENT_5,  0) +
             COALESCE(FLAG_DOCUMENT_6,  0) + COALESCE(FLAG_DOCUMENT_7,  0) +
             COALESCE(FLAG_DOCUMENT_8,  0) + COALESCE(FLAG_DOCUMENT_9,  0) +
             COALESCE(FLAG_DOCUMENT_10, 0) + COALESCE(FLAG_DOCUMENT_11, 0) +
             COALESCE(FLAG_DOCUMENT_12, 0) + COALESCE(FLAG_DOCUMENT_13, 0) +
             COALESCE(FLAG_DOCUMENT_14, 0) + COALESCE(FLAG_DOCUMENT_15, 0) +
             COALESCE(FLAG_DOCUMENT_16, 0) + COALESCE(FLAG_DOCUMENT_17, 0) +
             COALESCE(FLAG_DOCUMENT_18, 0) + COALESCE(FLAG_DOCUMENT_19, 0) +
             COALESCE(FLAG_DOCUMENT_20, 0) + COALESCE(FLAG_DOCUMENT_21, 0)
            )                                       AS TOTAL_DOCS_PROVIDED,
            -- Has provided at least one document
            CASE WHEN (
                COALESCE(FLAG_DOCUMENT_2,  0) + COALESCE(FLAG_DOCUMENT_3,  0) +
                COALESCE(FLAG_DOCUMENT_4,  0) + COALESCE(FLAG_DOCUMENT_5,  0) +
                COALESCE(FLAG_DOCUMENT_6,  0) + COALESCE(FLAG_DOCUMENT_7,  0) +
                COALESCE(FLAG_DOCUMENT_8,  0) + COALESCE(FLAG_DOCUMENT_9,  0) +
                COALESCE(FLAG_DOCUMENT_10, 0) + COALESCE(FLAG_DOCUMENT_11, 0) +
                COALESCE(FLAG_DOCUMENT_12, 0) + COALESCE(FLAG_DOCUMENT_13, 0) +
                COALESCE(FLAG_DOCUMENT_14, 0) + COALESCE(FLAG_DOCUMENT_15, 0) +
                COALESCE(FLAG_DOCUMENT_16, 0) + COALESCE(FLAG_DOCUMENT_17, 0) +
                COALESCE(FLAG_DOCUMENT_18, 0) + COALESCE(FLAG_DOCUMENT_19, 0) +
                COALESCE(FLAG_DOCUMENT_20, 0) + COALESCE(FLAG_DOCUMENT_21, 0)
            ) > 0 THEN 1 ELSE 0 END                     AS HAS_ANY_DOCUMENT,

            -- Total contact channels available (more = more verifiable client)
            COALESCE(FLAG_MOBIL,      0) +
            COALESCE(FLAG_EMP_PHONE,  0) +
            COALESCE(FLAG_WORK_PHONE, 0) +
            COALESCE(FLAG_CONT_MOBILE,0) +
            COALESCE(FLAG_PHONE,      0) +
            COALESCE(FLAG_EMAIL,      0)                AS CONTACT_SCORE,

            -- Client lives and works in the same area
            CASE WHEN REG_CITY_NOT_WORK_CITY    = 0
                  AND REG_REGION_NOT_WORK_REGION = 0
                 THEN 1 ELSE 0
            END                                     AS LIVE_WORK_SAME_REGION,

            -- Any default in social circle
            CASE WHEN DEF_30_CNT_SOCIAL_CIRCLE > 0
                   OR DEF_60_CNT_SOCIAL_CIRCLE > 0
                 THEN 1 ELSE 0
            END                                     AS ANY_SOCIAL_CIRCLE_DEF,

            -- Ratio of social circle defaults vs observations
            ROUND(
                DEF_60_CNT_SOCIAL_CIRCLE /
                NULLIF(OBS_60_CNT_SOCIAL_CIRCLE, 0), 4
            )                                       AS SOCIAL_CIRCLE_DEF_RATIO,

            -- Credit bureau enquiry pressure (recent enquiries = risky signal)
            COALESCE(AMT_REQ_CREDIT_BUREAU_HOUR, 0) +
            COALESCE(AMT_REQ_CREDIT_BUREAU_DAY,  0) +
            COALESCE(AMT_REQ_CREDIT_BUREAU_WEEK, 0) AS ENQUIRIES_SHORT_TERM,

            COALESCE(AMT_REQ_CREDIT_BUREAU_MON,  0) +
            COALESCE(AMT_REQ_CREDIT_BUREAU_QRT,  0) +
            COALESCE(AMT_REQ_CREDIT_BUREAU_YEAR, 0) AS ENQUIRIES_LONG_TERM,

            -- ── 5. Categorical encoding ───────────────────────────────
            -- Binary direct encoding
            CASE WHEN FLAG_OWN_CAR    = 'Y' THEN 1 ELSE 0 END AS HAS_CAR,
            CASE WHEN FLAG_OWN_REALTY = 'Y' THEN 1 ELSE 0 END AS HAS_REALTY,
            CASE WHEN CODE_GENDER     = 'M' THEN 1
                 WHEN CODE_GENDER     = 'F' THEN 0
                 ELSE NULL END                      AS IS_MALE,

            -- Ordinal encoding (order is meaningful)
            CASE NAME_EDUCATION_TYPE
                WHEN 'Lower secondary'          THEN 1
                WHEN 'Secondary / secondary special' THEN 2
                WHEN 'Incomplete higher'        THEN 3
                WHEN 'Higher education'         THEN 4
                WHEN 'Academic degree'          THEN 5
                ELSE NULL
            END                                     AS EDUCATION_LEVEL,

            REGION_RATING_CLIENT,
            REGION_RATING_CLIENT_W_CITY,

            -- One-hot for low cardinality nominals
            CASE WHEN NAME_CONTRACT_TYPE = 'Cash loans'      THEN 1 ELSE 0 END
                                                    AS IS_CASH_LOAN,
            CASE WHEN NAME_INCOME_TYPE = 'Working'           THEN 1 ELSE 0 END
                                                    AS INCOME_WORKING,
            CASE WHEN NAME_INCOME_TYPE = 'State servant'     THEN 1 ELSE 0 END
                                                    AS INCOME_STATE_SERVANT,
            CASE WHEN NAME_INCOME_TYPE = 'Pensioner'         THEN 1 ELSE 0 END
                                                    AS INCOME_PENSIONER,
            CASE WHEN NAME_INCOME_TYPE = 'Commercial associate' THEN 1 ELSE 0 END
                                                    AS INCOME_COMMERCIAL,

            CASE WHEN NAME_HOUSING_TYPE = 'House / apartment' THEN 1 ELSE 0 END
                                                    AS HOUSING_HOUSE,
            CASE WHEN NAME_HOUSING_TYPE = 'With parents'     THEN 1 ELSE 0 END
                                                    AS HOUSING_WITH_PARENTS,
            CASE WHEN NAME_HOUSING_TYPE = 'Municipal apartment' THEN 1 ELSE 0 END
                                                    AS HOUSING_MUNICIPAL,

            CASE WHEN NAME_FAMILY_STATUS = 'Married'         THEN 1 ELSE 0 END
                                                    AS IS_MARRIED,
            CASE WHEN NAME_FAMILY_STATUS = 'Single / not married' THEN 1 ELSE 0 END
                                                    AS IS_SINGLE,

            -- Pass through remaining raw columns for target encoding later
            -- (OCCUPATION_TYPE, ORGANIZATION_TYPE — high cardinality,
            --  handled in the modelling pipeline with cross-val target encoder)
            OCCUPATION_TYPE,
            ORGANIZATION_TYPE,

            -- Pass through all other raw numerics
            AMT_INCOME_TOTAL,
            AMT_CREDIT,
            AMT_ANNUITY,
            AMT_GOODS_PRICE,
            DAYS_BIRTH,
            DAYS_REGISTRATION,
            DAYS_ID_PUBLISH,
            DAYS_LAST_PHONE_CHANGE,
            DAYS_EMPLOYED,
            CNT_CHILDREN,
            CNT_FAM_MEMBERS,
            HOUR_APPR_PROCESS_START,
            DEF_30_CNT_SOCIAL_CIRCLE,
            DEF_60_CNT_SOCIAL_CIRCLE,
            OBS_30_CNT_SOCIAL_CIRCLE,
            OBS_60_CNT_SOCIAL_CIRCLE,
            REGION_POPULATION_RELATIVE,
            REG_CITY_NOT_WORK_CITY,
            REG_CITY_NOT_LIVE_CITY,
            REG_REGION_NOT_WORK_REGION,
            LIVE_CITY_NOT_WORK_CITY,
            LIVE_REGION_NOT_WORK_REGION,
            FLAG_EMP_PHONE,
            FLAG_WORK_PHONE,
            FLAG_CONT_MOBILE,
            FLAG_PHONE,
            FLAG_EMAIL,
            FLAG_DOCUMENT_3,
            FLAG_DOCUMENT_5,
            FLAG_DOCUMENT_6,
            FLAG_DOCUMENT_8,
            FLAG_DOCUMENT_16,
            FLAG_DOCUMENT_18

        FROM app_train_clean
    """)

    print("View 'app_train_features' created.")

build_main_table_features(con)

View 'app_train_features' created.


In [19]:
def sanity_check_features(con):
    """
    Quick checks on the new engineered features:
      - No division-by-zero producing INF
      - Ratios are in plausible ranges
      - Flag encodings are strictly 0/1
      - EXT_SOURCE combinations are bounded
    """
    checks = {
        "Rows in app_train_features":
            "SELECT COUNT(*) FROM app_train_features",

        "N new features created":
            "SELECT COUNT(*) FROM information_schema.columns \
             WHERE table_name = 'app_train_features'",

        "INF in CREDIT_INCOME_RATIO":
            "SELECT COUNT(*) FROM app_train_features \
             WHERE CREDIT_INCOME_RATIO = 'Infinity'::DOUBLE",

        "INF in ANNUITY_INCOME_RATIO":
            "SELECT COUNT(*) FROM app_train_features \
             WHERE ANNUITY_INCOME_RATIO = 'Infinity'::DOUBLE",

        "CREDIT_TERM max (months)":
            "SELECT ROUND(MAX(CREDIT_TERM), 0) FROM app_train_features",

        "AGE_YEARS range [min, max]":
            "SELECT CONCAT(ROUND(MIN(AGE_YEARS),1), ' — ', \
                           ROUND(MAX(AGE_YEARS),1)) \
             FROM app_train_features",

        "EXT_SOURCE_MEAN nulls":
            "SELECT COUNT(*) FROM app_train_features \
             WHERE EXT_SOURCE_MEAN IS NULL",

        "IS_MALE distinct values":
            "SELECT STRING_AGG(DISTINCT CAST(IS_MALE AS VARCHAR), ', ') \
             FROM app_train_features",

        "EDUCATION_LEVEL distinct values":
            "SELECT STRING_AGG(DISTINCT CAST(EDUCATION_LEVEL AS VARCHAR), ', ' \
             ORDER BY CAST(EDUCATION_LEVEL AS VARCHAR)) \
             FROM app_train_features",

        "TOTAL_DOCS_PROVIDED max":
            "SELECT MAX(TOTAL_DOCS_PROVIDED) FROM app_train_features",
    }

    print("Sanity checks on app_train_features\n")
    print(f"{'Check':<45} {'Result':>20}")
    print("─" * 68)
    for label, query in checks.items():
        result = con.execute(query).fetchone()[0]
        status = "  ⚠ FAIL" if result in [None, 'Infinity', 'inf'] else "  ✓"
        print(f"{label:<45} {str(result):>20}{status}")

sanity_check_features(con)

Sanity checks on app_train_features

Check                                                       Result
────────────────────────────────────────────────────────────────────
Rows in app_train_features                                  307511  ✓
N new features created                                          96  ✓
INF in CREDIT_INCOME_RATIO                                       0  ✓
INF in ANNUITY_INCOME_RATIO                                      0  ✓
CREDIT_TERM max (months)                                      45.0  ✓
AGE_YEARS range [min, max]                             20.5 — 69.1  ✓
EXT_SOURCE_MEAN nulls                                          172  ✓
IS_MALE distinct values                                       0, 1  ✓
EDUCATION_LEVEL distinct values                      1, 2, 3, 4, 5  ✓
TOTAL_DOCS_PROVIDED max                                          4  ✓


## Block 3 — Aggregations from Secondary Tables

The secondary tables contain historical behavioural data that is far more 
predictive than the static snapshot in `app_train`. However, they cannot be 
joined directly — each has a N:1 relationship with the main table and must 
first be aggregated at `SK_ID_CURR` level.

**One table is created per source, then joined in Block 3g:**

| Step | Table created | Source | Join key |
|---|---|---|---|
| 3a | `bureau_features` | `bureau` | `SK_ID_CURR` |
| 3b | `bureau_bal_features` | `bureau_balance` | `SK_ID_BUREAU` → `SK_ID_CURR` |
| 3c | `prev_app_features` | `prev_app` | `SK_ID_CURR` |
| 3d | `pos_features` | `pos_cash` | `SK_ID_CURR` |
| 3e | `installments_features` | `installments` | `SK_ID_CURR` |
| 3f | `cc_features` | `credit_card` | `SK_ID_CURR` |
| 3g | `app_train_final` | all of the above | `SK_ID_CURR` |

**Design decisions applied consistently:**
- All aggregations produce exactly one row per `SK_ID_CURR`
- `NULLIF` protects all ratio denominators from division by zero
- Clients with no records in a secondary table receive `NULL` for all 
  its features — handled via `LEFT JOIN` in step 3g and natively by LightGBM
- Recency windows (last 6/12 months) are computed where a time dimension 
  is available — recent behaviour outweighs distant history
- All tables are materialized to disk (not views) to avoid memory errors 
  on large secondary tables (up to 27M rows)

### 3a — bureau

**What it captures**: past external loan history from credit bureaus — loans taken with other financial institutions before the current application.

**Key signals**: number of active vs closed loans, total debt exposure, overdue amounts, loan prolongations (extensions = repayment difficulty signal).

**Coverage**: ~70% of clients have bureau history. The remaining ~30% have no external credit record — their NULL features are informative in themselve (no credit history = higher uncertainty for the lender).

**Most predictive features**: `BUREAU_DEBT_CREDIT_RATIO`, `BUREAU_EVER_OVERDUE`, `BUREAU_MAX_DAY_OVERDUE`, `BUREAU_ACTIVE_RATIO`.

In [7]:
def build_bureau_features(con):
    """
    Block 3a — Aggregations from bureau table.
    One row per SK_ID_CURR — ready to join with app_train_features.

    Features created:
      - Loan counts (total, active, closed)
      - Credit amounts (sum, mean, max)
      - Overdue amounts and days
      - Debt-to-credit ratio
      - Credit enquiry recency
    """
    con.execute("""
        CREATE OR REPLACE VIEW bureau_features AS
        SELECT
            SK_ID_CURR,

            -- ── Loan counts ───────────────────────────────────────────
            COUNT(*)                                        AS BUREAU_LOAN_COUNT,
            SUM(CASE WHEN CREDIT_ACTIVE = 'Active'
                     THEN 1 ELSE 0 END)                    AS BUREAU_ACTIVE_COUNT,
            SUM(CASE WHEN CREDIT_ACTIVE = 'Closed'
                     THEN 1 ELSE 0 END)                    AS BUREAU_CLOSED_COUNT,
            ROUND(
                SUM(CASE WHEN CREDIT_ACTIVE = 'Active' THEN 1 ELSE 0 END)
                / NULLIF(COUNT(*), 0)
            , 4)                                           AS BUREAU_ACTIVE_RATIO,

            -- ── Credit amounts ────────────────────────────────────────
            SUM(AMT_CREDIT_SUM)                            AS BUREAU_CREDIT_SUM,
            ROUND(AVG(AMT_CREDIT_SUM), 2)                  AS BUREAU_CREDIT_MEAN,
            MAX(AMT_CREDIT_SUM)                            AS BUREAU_CREDIT_MAX,
            SUM(AMT_CREDIT_SUM_DEBT)                       AS BUREAU_DEBT_SUM,
            ROUND(AVG(AMT_CREDIT_SUM_DEBT), 2)             AS BUREAU_DEBT_MEAN,

            -- ── Overdue ───────────────────────────────────────────────
            SUM(AMT_CREDIT_SUM_OVERDUE)                    AS BUREAU_OVERDUE_SUM,
            ROUND(AVG(AMT_CREDIT_SUM_OVERDUE), 2)          AS BUREAU_OVERDUE_MEAN,
            MAX(CREDIT_DAY_OVERDUE)                        AS BUREAU_MAX_DAY_OVERDUE,
            ROUND(AVG(CREDIT_DAY_OVERDUE), 2)              AS BUREAU_MEAN_DAY_OVERDUE,
            SUM(CASE WHEN CREDIT_DAY_OVERDUE > 0
                     THEN 1 ELSE 0 END)                    AS BUREAU_OVERDUE_COUNT,

            -- ── Debt-to-credit ratio ──────────────────────────────────
            -- How much of total credit is still owed
            ROUND(
                SUM(AMT_CREDIT_SUM_DEBT)
                / NULLIF(SUM(AMT_CREDIT_SUM), 0)
            , 4)                                           AS BUREAU_DEBT_CREDIT_RATIO,

            -- ── Temporal features ─────────────────────────────────────
            -- How long ago did the most recent bureau loan start
            MIN(DAYS_CREDIT)                               AS BUREAU_DAYS_CREDIT_MIN,
            ROUND(AVG(DAYS_CREDIT), 2)                     AS BUREAU_DAYS_CREDIT_MEAN,
            -- How far in the future is the latest end date
            MAX(DAYS_CREDIT_ENDDATE)                       AS BUREAU_ENDDATE_MAX,
            -- How long has the average loan been running
            ROUND(AVG(DAYS_CREDIT_UPDATE), 2)              AS BUREAU_DAYS_UPDATE_MEAN,

            -- ── Prolong count ─────────────────────────────────────────
            -- Loans that were extended — sign of repayment difficulty
            SUM(CNT_CREDIT_PROLONG)                        AS BUREAU_PROLONG_SUM,
            ROUND(AVG(CNT_CREDIT_PROLONG), 4)              AS BUREAU_PROLONG_MEAN,

            -- ── Credit type dummies (top types only) ──────────────────
            SUM(CASE WHEN CREDIT_TYPE = 'Consumer credit'
                     THEN 1 ELSE 0 END)                    AS BUREAU_CONSUMER_COUNT,
            SUM(CASE WHEN CREDIT_TYPE = 'Credit card'
                     THEN 1 ELSE 0 END)                    AS BUREAU_CARD_COUNT,
            SUM(CASE WHEN CREDIT_TYPE = 'Mortgage'
                     THEN 1 ELSE 0 END)                    AS BUREAU_MORTGAGE_COUNT,
            SUM(CASE WHEN CREDIT_TYPE = 'Car loan'
                     THEN 1 ELSE 0 END)                    AS BUREAU_CAR_COUNT,

            -- ── Binary flags ──────────────────────────────────────────
            MAX(CASE WHEN CREDIT_ACTIVE = 'Bad debt'
                     THEN 1 ELSE 0 END)                    AS BUREAU_HAS_BAD_DEBT,
            MAX(CASE WHEN AMT_CREDIT_SUM_OVERDUE > 0
                     THEN 1 ELSE 0 END)                    AS BUREAU_EVER_OVERDUE

        FROM bureau
        GROUP BY SK_ID_CURR
    """)
    print("View 'bureau_features' created.")


def sanity_check_bureau_features(con):
    checks = {
        "Rows in bureau_features":
            "SELECT COUNT(*) FROM bureau_features",

        "Clients in app_train with bureau data":
            """SELECT COUNT(DISTINCT b.SK_ID_CURR)
               FROM bureau_features b
               JOIN app_train a USING (SK_ID_CURR)""",

        "Clients in app_train WITHOUT bureau data":
            """SELECT COUNT(*)
               FROM app_train a
               LEFT JOIN bureau_features b USING (SK_ID_CURR)
               WHERE b.SK_ID_CURR IS NULL""",

        "BUREAU_LOAN_COUNT max":
            "SELECT MAX(BUREAU_LOAN_COUNT) FROM bureau_features",

        "BUREAU_LOAN_COUNT mean":
            "SELECT ROUND(AVG(BUREAU_LOAN_COUNT), 2) FROM bureau_features",

        "BUREAU_DEBT_CREDIT_RATIO > 1 (impossible)":
            """SELECT COUNT(*) FROM bureau_features
               WHERE BUREAU_DEBT_CREDIT_RATIO > 1""",

        "BUREAU_HAS_BAD_DEBT count":
            "SELECT SUM(BUREAU_HAS_BAD_DEBT) FROM bureau_features",

        "BUREAU_EVER_OVERDUE count":
            "SELECT SUM(BUREAU_EVER_OVERDUE) FROM bureau_features",

        "NULL rows in BUREAU_CREDIT_SUM":
            """SELECT COUNT(*) FROM bureau_features
               WHERE BUREAU_CREDIT_SUM IS NULL""",
    }

    print("Sanity checks on bureau_features\n")
    print(f"{'Check':<50} {'Result':>15}")
    print("─" * 68)
    for label, query in checks.items():
        result = con.execute(query).fetchone()[0]
        try:
            print(f"{label:<50} {result:>15,}  ✓")
        except (ValueError, TypeError):
            print(f"{label:<50} {str(result):>15}  ✓")


build_bureau_features(con)
sanity_check_bureau_features(con)

View 'bureau_features' created.
Sanity checks on bureau_features

Check                                                       Result
────────────────────────────────────────────────────────────────────
Rows in bureau_features                                    305,811  ✓
Clients in app_train with bureau data                      263,491  ✓
Clients in app_train WITHOUT bureau data                    44,020  ✓
BUREAU_LOAN_COUNT max                                          116  ✓
BUREAU_LOAN_COUNT mean                                        5.61  ✓
BUREAU_DEBT_CREDIT_RATIO > 1 (impossible)                    1,751  ✓
BUREAU_HAS_BAD_DEBT count                                       21  ✓
BUREAU_EVER_OVERDUE count                                    3,801  ✓
NULL rows in BUREAU_CREDIT_SUM                                   2  ✓


### 3b — bureau_balance

**What it captures**: month-by-month payment status for each external loan — the most granular view of historical repayment behaviour available.

**Key signals**: DPD (days past due) history, ratio of months with late payments, trend between recent and historical behaviour.

**Note on join**: `bureau_balance` does not contain `SK_ID_CURR` directly — it must be joined through `bureau` via `SK_ID_BUREAU` first.

**Status codes**: C = closed, X = unknown, 0 = on time, 1-5 = months past due.

**Most predictive features**: `BBAL_DPD_MAX`, `BBAL_EVER_LATE`, `BBAL_DPD_TREND` (positive = worsening behaviour).

In [8]:
def build_bureau_balance_features(con):
    """
    Block 3b — Aggregations from bureau_balance table.
    
    bureau_balance has two join levels:
      bureau_balance → bureau (via SK_ID_BUREAU)
      bureau         → app_train (via SK_ID_CURR)
    
    So we must join through bureau to get to SK_ID_CURR.

    STATUS codes:
      C = closed, X = unknown, 0 = no DPD, 1-5 = months past due
    """
    con.execute("""
        CREATE OR REPLACE VIEW bureau_bal_features AS
        WITH base AS (
            SELECT
                b.SK_ID_CURR,
                bb.MONTHS_BALANCE,
                bb.STATUS,
                CASE WHEN bb.STATUS IN ('C', 'X') THEN NULL
                     ELSE CAST(bb.STATUS AS INTEGER)
                END AS DPD
            FROM bureau_balance bb
            JOIN bureau b USING (SK_ID_BUREAU)
        )
        SELECT
            SK_ID_CURR,

            -- ── Volume ────────────────────────────────────────────────
            COUNT(*)                                        AS BBAL_MONTHS_COUNT,
            COUNT(DISTINCT MONTHS_BALANCE)                  AS BBAL_UNIQUE_MONTHS,

            -- ── Status distribution ───────────────────────────────────
            ROUND(AVG(CASE WHEN STATUS = 'C'
                          THEN 1.0 ELSE 0 END), 4)         AS BBAL_STATUS_C_RATIO,
            ROUND(AVG(CASE WHEN STATUS = '0'
                          THEN 1.0 ELSE 0 END), 4)         AS BBAL_STATUS_0_RATIO,
            ROUND(AVG(CASE WHEN STATUS NOT IN ('C', 'X', '0')
                          THEN 1.0 ELSE 0 END), 4)         AS BBAL_DPD_RATIO,

            -- ── DPD statistics ────────────────────────────────────────
            MAX(DPD)                                        AS BBAL_DPD_MAX,
            ROUND(AVG(DPD), 4)                             AS BBAL_DPD_MEAN,
            SUM(CASE WHEN DPD > 0 THEN 1 ELSE 0 END)       AS BBAL_DPD_COUNT,

            -- ── Ever late flag ────────────────────────────────────────
            MAX(CASE WHEN DPD > 0 THEN 1 ELSE 0 END)       AS BBAL_EVER_LATE,
            MAX(CASE WHEN DPD >= 3 THEN 1 ELSE 0 END)       AS BBAL_EVER_LATE_3M,

            -- ── Recency — last 12 months only ─────────────────────────
            -- MONTHS_BALANCE is negative: -1 = last month, -12 = 12 months ago
            ROUND(AVG(CASE WHEN MONTHS_BALANCE >= -12 AND DPD IS NOT NULL
                          THEN DPD END), 4)                AS BBAL_DPD_MEAN_12M,
            MAX(CASE WHEN MONTHS_BALANCE >= -12
                     THEN DPD END)                         AS BBAL_DPD_MAX_12M,
            ROUND(AVG(CASE WHEN MONTHS_BALANCE >= -6 AND DPD IS NOT NULL
                          THEN DPD END), 4)                AS BBAL_DPD_MEAN_6M,

            -- ── Trend — is behaviour improving or worsening? ──────────
            -- Positive = worsening (recent DPD > historical DPD)
            ROUND(
                AVG(CASE WHEN MONTHS_BALANCE >= -6  AND DPD IS NOT NULL THEN DPD END) -
                AVG(CASE WHEN MONTHS_BALANCE <  -6  AND DPD IS NOT NULL THEN DPD END)
            , 4)                                           AS BBAL_DPD_TREND

        FROM base
        GROUP BY SK_ID_CURR
    """)
    print("View 'bureau_bal_features' created.")


def sanity_check_bureau_bal_features(con):
    checks = {
        "Rows in bureau_bal_features":
            "SELECT COUNT(*) FROM bureau_bal_features",

        "Clients with bureau_balance data":
            """SELECT COUNT(DISTINCT b.SK_ID_CURR)
               FROM bureau_bal_features b
               JOIN app_train a USING (SK_ID_CURR)""",

        "Clients WITHOUT bureau_balance data":
            """SELECT COUNT(*)
               FROM app_train a
               LEFT JOIN bureau_bal_features b USING (SK_ID_CURR)
               WHERE b.SK_ID_CURR IS NULL""",

        "BBAL_DPD_MAX max value (should be 0-5)":
            "SELECT MAX(BBAL_DPD_MAX) FROM bureau_bal_features",

        "BBAL_EVER_LATE count":
            "SELECT SUM(BBAL_EVER_LATE) FROM bureau_bal_features",

        "BBAL_EVER_LATE_3M count":
            "SELECT SUM(BBAL_EVER_LATE_3M) FROM bureau_bal_features",

        "BBAL_STATUS_C_RATIO > 1 (impossible)":
            """SELECT COUNT(*) FROM bureau_bal_features
               WHERE BBAL_STATUS_C_RATIO > 1""",

        "BBAL_DPD_TREND nulls (no history on one side)":
            """SELECT COUNT(*) FROM bureau_bal_features
               WHERE BBAL_DPD_TREND IS NULL""",
    }

    print("Sanity checks on bureau_bal_features\n")
    print(f"{'Check':<50} {'Result':>15}")
    print("─" * 68)
    for label, query in checks.items():
        result = con.execute(query).fetchone()[0]
        try:
            print(f"{label:<50} {result:>15,}  ✓")
        except (ValueError, TypeError):
            print(f"{label:<50} {str(result):>15}  ✓")


build_bureau_balance_features(con)
sanity_check_bureau_bal_features(con)

View 'bureau_bal_features' created.
Sanity checks on bureau_bal_features

Check                                                       Result
────────────────────────────────────────────────────────────────────


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows in bureau_bal_features                                134,542  ✓


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Clients with bureau_balance data                            92,231  ✓


...

BBAL_EVER_LATE_3M count                                      4,829  ✓


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

BBAL_STATUS_C_RATIO > 1 (impossible)                             0  ✓


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

BBAL_DPD_TREND nulls (no history on one side)               38,258  ✓


### 3c — previous_application

**What it captures**: all past loan applications made by the client directly to Home Credit — both approved and refused.

**Key signals**: approval/refusal rate, credit amounts requested vs granted, number of past refusals, most recent application outcome.

**Preprocessing**: rows with `NAME_CONTRACT_STATUS = 'Unused offer'` are excluded — these are pre-approvals the client never acted on and would distort counts and ratios.

**Most predictive features**: `PREV_REFUSAL_RATE`, `PREV_EVER_REFUSED`, `PREV_APPROVAL_RATE`, `PREV_CREDIT_APPLICATION_RATIO`.

In [9]:
def build_prev_app_features(con):
    """
    Block 3c — Aggregations from prev_app table.
    One row per SK_ID_CURR — ready to join with app_train_features.

    Features created:
      - Application counts (total, approved, refused, cancelled)
      - Approval rate and refusal rate
      - Credit and annuity amounts (mean, max)
      - Down payment statistics
      - Loan purpose dummies
      - Recency features (last application)
    """
    con.execute("""
        CREATE OR REPLACE VIEW prev_app_features AS
        WITH base AS (
            SELECT
                SK_ID_CURR,
                SK_ID_PREV,
                NAME_CONTRACT_STATUS,
                AMT_CREDIT,
                AMT_ANNUITY,
                AMT_DOWN_PAYMENT,
                AMT_APPLICATION,
                CNT_PAYMENT,
                DAYS_DECISION,
                NAME_CONTRACT_TYPE,
                NAME_CASH_LOAN_PURPOSE,
                NAME_YIELD_GROUP,
                RATE_DOWN_PAYMENT,
                NFLAG_INSURED_ON_APPROVAL,
                RATE_INTEREST_PRIMARY,   
                -- Flag for most recent application per client
                ROW_NUMBER() OVER (
                    PARTITION BY SK_ID_CURR
                    ORDER BY DAYS_DECISION DESC
                ) AS recency_rank
            FROM prev_app
            WHERE NAME_CONTRACT_STATUS != 'Unused offer'
        )
        SELECT
            SK_ID_CURR,

            -- ── Application counts ────────────────────────────────────
            COUNT(*)                                        AS PREV_APP_COUNT,
            SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Approved'
                     THEN 1 ELSE 0 END)                    AS PREV_APPROVED_COUNT,
            SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Refused'
                     THEN 1 ELSE 0 END)                    AS PREV_REFUSED_COUNT,
            SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Canceled'
                     THEN 1 ELSE 0 END)                    AS PREV_CANCELED_COUNT,

            -- ── Approval / refusal rates ──────────────────────────────
            ROUND(
                SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Approved' THEN 1 ELSE 0 END)
                / NULLIF(COUNT(*), 0)
            , 4)                                           AS PREV_APPROVAL_RATE,
            ROUND(
                SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Refused' THEN 1 ELSE 0 END)
                / NULLIF(COUNT(*), 0)
            , 4)                                           AS PREV_REFUSAL_RATE,

            -- ── Credit amounts ────────────────────────────────────────
            ROUND(AVG(AMT_CREDIT), 2)                      AS PREV_AMT_CREDIT_MEAN,
            MAX(AMT_CREDIT)                                AS PREV_AMT_CREDIT_MAX,
            ROUND(AVG(AMT_APPLICATION), 2)                 AS PREV_AMT_APPLICATION_MEAN,

            -- How much less was approved vs requested (discount ratio)
            ROUND(
                AVG(AMT_CREDIT / NULLIF(AMT_APPLICATION, 0))
            , 4)                                           AS PREV_CREDIT_APPLICATION_RATIO,

            -- ── Annuity ───────────────────────────────────────────────
            ROUND(AVG(AMT_ANNUITY), 2)                     AS PREV_AMT_ANNUITY_MEAN,
            MAX(AMT_ANNUITY)                               AS PREV_AMT_ANNUITY_MAX,

            -- ── Down payment ──────────────────────────────────────────
            ROUND(AVG(AMT_DOWN_PAYMENT), 2)                AS PREV_DOWN_PAYMENT_MEAN,
            ROUND(AVG(RATE_DOWN_PAYMENT), 4)               AS PREV_DOWN_PAYMENT_RATE_MEAN,

            -- ── Loan duration ─────────────────────────────────────────
            ROUND(AVG(CNT_PAYMENT), 2)                     AS PREV_CNT_PAYMENT_MEAN,
            MAX(CNT_PAYMENT)                               AS PREV_CNT_PAYMENT_MAX,

            -- ── Recency ───────────────────────────────────────────────
            -- Days since last application decision (less negative = more recent)
            MAX(DAYS_DECISION)                             AS PREV_DAYS_DECISION_LAST,
            ROUND(AVG(DAYS_DECISION), 2)                   AS PREV_DAYS_DECISION_MEAN,

            -- ── Contract type counts ──────────────────────────────────
            SUM(CASE WHEN NAME_CONTRACT_TYPE = 'Consumer loans'
                     THEN 1 ELSE 0 END)                    AS PREV_CONSUMER_COUNT,
            SUM(CASE WHEN NAME_CONTRACT_TYPE = 'Cash loans'
                     THEN 1 ELSE 0 END)                    AS PREV_CASH_COUNT,
            SUM(CASE WHEN NAME_CONTRACT_TYPE = 'Revolving loans'
                     THEN 1 ELSE 0 END)                    AS PREV_REVOLVING_COUNT,

            -- ── Last application features (recency signal) ────────────
            MAX(CASE WHEN recency_rank = 1
                     THEN NAME_CONTRACT_STATUS END)        AS PREV_LAST_STATUS,
            MAX(CASE WHEN recency_rank = 1
                     THEN AMT_CREDIT END)                  AS PREV_LAST_CREDIT,
            MAX(CASE WHEN recency_rank = 1
                     THEN CNT_PAYMENT END)                 AS PREV_LAST_CNT_PAYMENT,
                
            -- Insurance flag mean (higher = more insured clients = lower risk)
            ROUND(AVG(NFLAG_INSURED_ON_APPROVAL), 4)           AS PREV_NFLAG_INSURED_MEAN,

            -- Primary interest rate mean
            ROUND(AVG(RATE_INTEREST_PRIMARY), 4)               AS PREV_INTEREST_RATE_MEAN,

            -- ── Recent window — last 12 months only ──────────────────────
            -- DAYS_DECISION is negative: -365 = one year ago
            ROUND(
                AVG(CASE WHEN DAYS_DECISION >= -365
                        THEN AMT_CREDIT END), 2)              AS PREV_AMT_CREDIT_MEAN_12M,
            SUM(CASE WHEN DAYS_DECISION >= -365
                    THEN 1 ELSE 0 END)                        AS PREV_APP_COUNT_12M,
            ROUND(
                SUM(CASE WHEN DAYS_DECISION >= -365
                        AND NAME_CONTRACT_STATUS = 'Approved'
                        THEN 1 ELSE 0 END)
                / NULLIF(SUM(CASE WHEN DAYS_DECISION >= -365
                                THEN 1 ELSE 0 END), 0)
            , 4)                                               AS PREV_APPROVAL_RATE_12M,

            -- ── Aggregations by contract type ────────────────────────────
            ROUND(AVG(CASE WHEN NAME_CONTRACT_TYPE = 'Cash loans'
                        THEN AMT_CREDIT END), 2)            AS PREV_CASH_CREDIT_MEAN,
            ROUND(AVG(CASE WHEN NAME_CONTRACT_TYPE = 'Revolving loans'
                        THEN AMT_CREDIT END), 2)            AS PREV_REVOLVING_CREDIT_MEAN,
            ROUND(AVG(CASE WHEN NAME_CONTRACT_TYPE = 'Consumer loans'
                        THEN AMT_CREDIT END), 2)            AS PREV_CONSUMER_CREDIT_MEAN,
            ROUND(AVG(CASE WHEN NAME_CONTRACT_TYPE = 'Cash loans'
                        THEN AMT_ANNUITY END), 2)           AS PREV_CASH_ANNUITY_MEAN,
            ROUND(AVG(CASE WHEN NAME_CONTRACT_TYPE = 'Revolving loans'
                        THEN AMT_ANNUITY END), 2)           AS PREV_REVOLVING_ANNUITY_MEAN,

            -- ── Binary flags ──────────────────────────────────────────
            MAX(CASE WHEN NAME_CONTRACT_STATUS = 'Refused'
                     THEN 1 ELSE 0 END)                    AS PREV_EVER_REFUSED,
            MAX(CASE WHEN NAME_YIELD_GROUP = 'high'
                     THEN 1 ELSE 0 END)                    AS PREV_EVER_HIGH_YIELD

        FROM base
        GROUP BY SK_ID_CURR
    """)
    print("View 'prev_app_features' created.")


def sanity_check_prev_app_features(con):
    checks = {
        "Rows in prev_app_features":
            "SELECT COUNT(*) FROM prev_app_features",

        "Clients in app_train with prev_app data":
            """SELECT COUNT(DISTINCT p.SK_ID_CURR)
               FROM prev_app_features p
               JOIN app_train a USING (SK_ID_CURR)""",

        "Clients in app_train WITHOUT prev_app data":
            """SELECT COUNT(*)
               FROM app_train a
               LEFT JOIN prev_app_features p USING (SK_ID_CURR)
               WHERE p.SK_ID_CURR IS NULL""",

        "PREV_APP_COUNT max":
            "SELECT MAX(PREV_APP_COUNT) FROM prev_app_features",

        "PREV_APP_COUNT mean":
            "SELECT ROUND(AVG(PREV_APP_COUNT), 2) FROM prev_app_features",

        "PREV_APPROVAL_RATE range [min, max]":
            """SELECT CONCAT(
                ROUND(MIN(PREV_APPROVAL_RATE), 2), ' — ',
                ROUND(MAX(PREV_APPROVAL_RATE), 2)
               ) FROM prev_app_features""",

        "PREV_EVER_REFUSED count":
            "SELECT SUM(PREV_EVER_REFUSED) FROM prev_app_features",

        "PREV_LAST_STATUS distinct values":
            """SELECT STRING_AGG(DISTINCT PREV_LAST_STATUS, ', ')
               FROM prev_app_features""",

        "PREV_CREDIT_APPLICATION_RATIO > 2 (suspicious)":
            """SELECT COUNT(*) FROM prev_app_features
               WHERE PREV_CREDIT_APPLICATION_RATIO > 2""",
    }

    print("Sanity checks on prev_app_features\n")
    print(f"{'Check':<50} {'Result':>20}")
    print("─" * 73)
    for label, query in checks.items():
        result = con.execute(query).fetchone()[0]
        try:
            print(f"{label:<50} {result:>20,}  ✓")
        except (ValueError, TypeError):
            print(f"{label:<50} {str(result):>20}  ✓")


build_prev_app_features(con)
sanity_check_prev_app_features(con)

View 'prev_app_features' created.
Sanity checks on prev_app_features

Check                                                            Result
─────────────────────────────────────────────────────────────────────────


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows in prev_app_features                                       338,682  ✓


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Clients in app_train with prev_app data                         290,910  ✓


...

Clients in app_train WITHOUT prev_app data                       16,601  ✓


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

PREV_APP_COUNT max                                                   77  ✓
PREV_APP_COUNT mean                                                4.85  ✓


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

PREV_APPROVAL_RATE range [min, max]                           0.0 — 1.0  ✓
PREV_EVER_REFUSED count                                         118,277  ✓
PREV_LAST_STATUS distinct values                   Approved, Refused, Canceled  ✓
PREV_CREDIT_APPLICATION_RATIO > 2 (suspicious)                       45  ✓


### 3d — POS_CASH_balance

**What it captures**: monthly snapshots of past POS (point-of-sale) and cash loan contracts at Home Credit.

**Key signals**: DPD history on past HC contracts, completion rate, number of active contracts, recent vs historical DPD trend.

**Note on DPD**: two DPD measures are available — `SK_DPD` (raw days past due) and `SK_DPD_DEF` (adjusted, more conservative). Both are aggregated as they capture slightly different aspects of payment behaviour.

**Most predictive features**: `POS_DPD_MAX`, `POS_EVER_LATE_60`, `POS_DPD_TREND`, `POS_COMPLETED_RATIO`.

In [10]:
def build_pos_features(con):
    """
    Block 3d — Aggregations from pos_cash table.
    One row per SK_ID_CURR — ready to join with app_train_features.

    Features created:
      - Contract counts and completion rate
      - DPD statistics (mean, max, recency)
      - Trend signal (improving vs worsening)
    """
    con.execute("""
        CREATE OR REPLACE VIEW pos_features AS
        WITH base AS (
            SELECT
                SK_ID_CURR,
                SK_ID_PREV,
                MONTHS_BALANCE,
                NAME_CONTRACT_STATUS,
                SK_DPD,
                SK_DPD_DEF,
                CNT_INSTALMENT,
                CNT_INSTALMENT_FUTURE
            FROM pos_cash
        )
        SELECT
            SK_ID_CURR,

            -- ── Volume ────────────────────────────────────────────────
            COUNT(*)                                        AS POS_MONTHS_COUNT,
            COUNT(DISTINCT SK_ID_PREV)                      AS POS_CONTRACT_COUNT,

            -- ── Completion ───────────────────────────────────────────
            -- Completed = no future instalments remaining
            ROUND(
                SUM(CASE WHEN CNT_INSTALMENT_FUTURE = 0
                         THEN 1 ELSE 0 END)
                / NULLIF(COUNT(*), 0)
            , 4)                                           AS POS_COMPLETED_RATIO,
            MAX(CASE WHEN NAME_CONTRACT_STATUS = 'Completed'
                     THEN 1 ELSE 0 END)                    AS POS_EVER_COMPLETED,
            SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Active'
                     THEN 1 ELSE 0 END)                    AS POS_ACTIVE_COUNT,

            -- ── DPD statistics ────────────────────────────────────────
            MAX(SK_DPD)                                     AS POS_DPD_MAX,
            ROUND(AVG(SK_DPD), 4)                          AS POS_DPD_MEAN,
            MAX(SK_DPD_DEF)                                 AS POS_DPD_DEF_MAX,
            ROUND(AVG(SK_DPD_DEF), 4)                      AS POS_DPD_DEF_MEAN,
            SUM(CASE WHEN SK_DPD > 0 THEN 1 ELSE 0 END)    AS POS_DPD_COUNT,
            
            -- Average total instalments per contract
            ROUND(AVG(CNT_INSTALMENT), 2)                      AS POS_CNT_INSTALMENT_MEAN,

            -- Average remaining instalments (proxy for how early in the loan)
            ROUND(AVG(CNT_INSTALMENT_FUTURE), 2)               AS POS_CNT_INSTALMENT_FUTURE_MEAN,

            -- ── Binary flags ──────────────────────────────────────────
            MAX(CASE WHEN SK_DPD > 0
                     THEN 1 ELSE 0 END)                    AS POS_EVER_LATE,
            MAX(CASE WHEN SK_DPD >= 30
                     THEN 1 ELSE 0 END)                    AS POS_EVER_LATE_30,
            MAX(CASE WHEN SK_DPD >= 60
                     THEN 1 ELSE 0 END)                    AS POS_EVER_LATE_60,

            -- ── Recency — last 12 and 6 months ───────────────────────
            ROUND(AVG(CASE WHEN MONTHS_BALANCE >= -12
                          THEN SK_DPD END), 4)             AS POS_DPD_MEAN_12M,
            MAX(CASE WHEN MONTHS_BALANCE >= -12
                     THEN SK_DPD END)                      AS POS_DPD_MAX_12M,
            ROUND(AVG(CASE WHEN MONTHS_BALANCE >= -6
                          THEN SK_DPD END), 4)             AS POS_DPD_MEAN_6M,
            SUM(CASE WHEN MONTHS_BALANCE >= -12
                      AND SK_DPD > 0
                     THEN 1 ELSE 0 END)                    AS POS_DPD_COUNT_12M,

            -- ── Trend ─────────────────────────────────────────────────
            ROUND(
                AVG(CASE WHEN MONTHS_BALANCE >= -6  THEN SK_DPD END) -
                AVG(CASE WHEN MONTHS_BALANCE <  -6  THEN SK_DPD END)
            , 4)                                           AS POS_DPD_TREND

        FROM base
        GROUP BY SK_ID_CURR
    """)
    print("View 'pos_features' created.")


def sanity_check_pos_features(con):
    checks = {
        "Rows in pos_features":
            "SELECT COUNT(*) FROM pos_features",

        "Clients in app_train with pos data":
            """SELECT COUNT(DISTINCT p.SK_ID_CURR)
               FROM pos_features p
               JOIN app_train a USING (SK_ID_CURR)""",

        "Clients in app_train WITHOUT pos data":
            """SELECT COUNT(*)
               FROM app_train a
               LEFT JOIN pos_features p USING (SK_ID_CURR)
               WHERE p.SK_ID_CURR IS NULL""",

        "POS_DPD_MAX max value":
            "SELECT MAX(POS_DPD_MAX) FROM pos_features",

        "POS_DPD_MEAN mean value":
            "SELECT ROUND(AVG(POS_DPD_MEAN), 4) FROM pos_features",

        "POS_EVER_LATE count":
            "SELECT SUM(POS_EVER_LATE) FROM pos_features",

        "POS_EVER_LATE_60 count":
            "SELECT SUM(POS_EVER_LATE_60) FROM pos_features",

        "POS_COMPLETED_RATIO > 1 (impossible)":
            """SELECT COUNT(*) FROM pos_features
               WHERE POS_COMPLETED_RATIO > 1""",

        "POS_DPD_TREND nulls":
            """SELECT COUNT(*) FROM pos_features
               WHERE POS_DPD_TREND IS NULL""",
    }

    print("Sanity checks on pos_features\n")
    print(f"{'Check':<50} {'Result':>15}")
    print("─" * 68)
    for label, query in checks.items():
        result = con.execute(query).fetchone()[0]
        try:
            print(f"{label:<50} {result:>15,}  ✓")
        except (ValueError, TypeError):
            print(f"{label:<50} {str(result):>15}  ✓")


build_pos_features(con)
sanity_check_pos_features(con)

View 'pos_features' created.
Sanity checks on pos_features

Check                                                       Result
────────────────────────────────────────────────────────────────────


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows in pos_features                                       337,252  ✓


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Clients in app_train with pos data                         289,444  ✓


...

POS_EVER_LATE_60 count                                       5,760  ✓


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

POS_COMPLETED_RATIO > 1 (impossible)                             0  ✓


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

POS_DPD_TREND nulls                                        149,277  ✓


### installments_payments

**What it captures**: instalment-level payment history for past HC loans — one row per scheduled payment, with actual payment date and amount.

**Key signals**: payment delay (days late/early), payment amount vs instalment amount (underpayment), late payment ratio, recency trend.

**Two complementary signals**:
- `INSTAL_LATE_RATIO` — how *often* the client pays late
- `INSTAL_DELAY_MEAN` — how *much* late on average  
A client can be frequently late but only by 1-2 days (low risk) vs rarely late but by 30+ days when it happens (high risk).

**Most predictive features**: `INSTAL_LATE_RATIO`, `INSTAL_DELAY_MAX`, `INSTAL_EVER_LATE_30`, `INSTAL_PAYMENT_RATIO_MEAN`.

In [11]:
def build_installments_features(con):

    # Step 1 — base table (già funzionava)
    print("Step 1/4 — computing base metrics...")
    con.execute("""
        CREATE OR REPLACE TABLE installments_base AS
        SELECT
            SK_ID_CURR,
            SK_ID_PREV,
            DAYS_INSTALMENT,
            DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT        AS PAYMENT_DELAY,
            AMT_PAYMENT / NULLIF(AMT_INSTALMENT, 0)     AS PAYMENT_RATIO,
            AMT_INSTALMENT,
            AMT_PAYMENT
        FROM installments
        WHERE DAYS_ENTRY_PAYMENT IS NOT NULL
          AND DAYS_INSTALMENT    IS NOT NULL
    """)
    print("Step 1/4 — done.")

    # Step 2a — counts and late stats only
    print("Step 2/4 — counts and late stats...")
    con.execute("""
        CREATE OR REPLACE TABLE instal_part1 AS
        SELECT
            SK_ID_CURR,
            COUNT(*)                                        AS INSTAL_COUNT,
            COUNT(DISTINCT SK_ID_PREV)                      AS INSTAL_CONTRACT_COUNT,
            SUM(CASE WHEN PAYMENT_DELAY > 0 THEN 1 ELSE 0 END) AS INSTAL_LATE_COUNT,
            ROUND(SUM(CASE WHEN PAYMENT_DELAY > 0 THEN 1 ELSE 0 END)
                / NULLIF(COUNT(*), 0), 4)                  AS INSTAL_LATE_RATIO,
            MAX(CASE WHEN PAYMENT_DELAY > 0 THEN 1 ELSE 0 END) AS INSTAL_EVER_LATE,
            MAX(CASE WHEN PAYMENT_DELAY > 30 THEN 1 ELSE 0 END) AS INSTAL_EVER_LATE_30,
            SUM(CASE WHEN PAYMENT_DELAY < 0 THEN 1 ELSE 0 END) AS INSTAL_EARLY_COUNT
        FROM installments_base
        GROUP BY SK_ID_CURR
    """)
    print("Step 2/4 — done.")

    # Step 2b — delay and amount stats
    print("Step 3/4 — delay and amount stats...")
    con.execute("""
        CREATE OR REPLACE TABLE instal_part2 AS
        SELECT
            SK_ID_CURR,
            ROUND(AVG(PAYMENT_DELAY), 2)                   AS INSTAL_DELAY_MEAN,
            MAX(PAYMENT_DELAY)                             AS INSTAL_DELAY_MAX,
            ROUND(AVG(CASE WHEN PAYMENT_DELAY > 0
                          THEN PAYMENT_DELAY END), 2)      AS INSTAL_DELAY_MEAN_LATE,
            ROUND(AVG(CASE WHEN PAYMENT_DELAY < 0
                          THEN ABS(PAYMENT_DELAY) END), 2) AS INSTAL_EARLY_DAYS_MEAN,
            ROUND(AVG(PAYMENT_RATIO), 4)                   AS INSTAL_PAYMENT_RATIO_MEAN,
            ROUND(MIN(PAYMENT_RATIO), 4)                   AS INSTAL_PAYMENT_RATIO_MIN,
            SUM(CASE WHEN PAYMENT_RATIO < 0.95
                     THEN 1 ELSE 0 END)                    AS INSTAL_UNDERPAID_COUNT,
            MAX(CASE WHEN PAYMENT_RATIO < 0.95
                     THEN 1 ELSE 0 END)                    AS INSTAL_EVER_UNDERPAID,
            ROUND(AVG(AMT_INSTALMENT), 2)                  AS INSTAL_AMT_MEAN,
            ROUND(AVG(AMT_PAYMENT), 2)                     AS INSTAL_PAYMENT_MEAN,
            SUM(AMT_INSTALMENT - AMT_PAYMENT)              AS INSTAL_AMT_DIFF_SUM
        FROM installments_base
        GROUP BY SK_ID_CURR
    """)
    print("Step 3/4 — done.")

    # Step 2c — recency and trend
    print("Step 4/4 — recency and trend...")
    con.execute("""
        CREATE OR REPLACE TABLE instal_part3 AS
        SELECT
            SK_ID_CURR,
            ROUND(AVG(CASE WHEN DAYS_INSTALMENT >= -365
                          THEN PAYMENT_DELAY END), 2)      AS INSTAL_DELAY_MEAN_12M,
            MAX(CASE WHEN DAYS_INSTALMENT >= -365
                     THEN PAYMENT_DELAY END)               AS INSTAL_DELAY_MAX_12M,
            ROUND(AVG(CASE WHEN DAYS_INSTALMENT >= -180
                          THEN PAYMENT_DELAY END), 2)      AS INSTAL_DELAY_MEAN_6M,
            ROUND(
                SUM(CASE WHEN DAYS_INSTALMENT >= -365
                          AND PAYMENT_DELAY > 0 THEN 1 ELSE 0 END)
                / NULLIF(SUM(CASE WHEN DAYS_INSTALMENT >= -365
                              THEN 1 ELSE 0 END), 0)
            , 4)                                           AS INSTAL_LATE_RATIO_12M,
            ROUND(
                AVG(CASE WHEN DAYS_INSTALMENT >= -180 THEN PAYMENT_DELAY END) -
                AVG(CASE WHEN DAYS_INSTALMENT <  -180 THEN PAYMENT_DELAY END)
            , 4)                                           AS INSTAL_DELAY_TREND
        FROM installments_base
        GROUP BY SK_ID_CURR
    """)
    print("Step 4/4 — done.")

    # Final join of all parts
    print("Joining parts...")
    con.execute("""
        CREATE OR REPLACE TABLE installments_features AS
        SELECT
            p1.*,
            p2.INSTAL_DELAY_MEAN,
            p2.INSTAL_DELAY_MAX,
            p2.INSTAL_DELAY_MEAN_LATE,
            p2.INSTAL_EARLY_DAYS_MEAN,
            p2.INSTAL_PAYMENT_RATIO_MEAN,
            p2.INSTAL_PAYMENT_RATIO_MIN,
            p2.INSTAL_UNDERPAID_COUNT,
            p2.INSTAL_EVER_UNDERPAID,
            p2.INSTAL_AMT_MEAN,
            p2.INSTAL_PAYMENT_MEAN,
            p2.INSTAL_AMT_DIFF_SUM,
            p3.INSTAL_DELAY_MEAN_12M,
            p3.INSTAL_DELAY_MAX_12M,
            p3.INSTAL_DELAY_MEAN_6M,
            p3.INSTAL_LATE_RATIO_12M,
            p3.INSTAL_DELAY_TREND
        FROM instal_part1 p1
        LEFT JOIN instal_part2 p2 USING (SK_ID_CURR)
        LEFT JOIN instal_part3 p3 USING (SK_ID_CURR)
    """)

    # Cleanup
    con.execute("DROP TABLE IF EXISTS installments_base")
    con.execute("DROP TABLE IF EXISTS instal_part1")
    con.execute("DROP TABLE IF EXISTS instal_part2")
    con.execute("DROP TABLE IF EXISTS instal_part3")
    print("Done. Table 'installments_features' created.")


def sanity_check_installments_features(con):
    checks = {
        "Rows in installments_features":
            "SELECT COUNT(*) FROM installments_features",

        "Clients in app_train with installments data":
            """SELECT COUNT(DISTINCT i.SK_ID_CURR)
               FROM installments_features i
               JOIN app_train a USING (SK_ID_CURR)""",

        "Clients in app_train WITHOUT installments data":
            """SELECT COUNT(*)
               FROM app_train a
               LEFT JOIN installments_features i USING (SK_ID_CURR)
               WHERE i.SK_ID_CURR IS NULL""",

        "INSTAL_COUNT mean":
            "SELECT ROUND(AVG(INSTAL_COUNT), 1) FROM installments_features",

        "INSTAL_LATE_RATIO range [min, max]":
            """SELECT CONCAT(
                ROUND(MIN(INSTAL_LATE_RATIO), 3), ' — ',
                ROUND(MAX(INSTAL_LATE_RATIO), 3)
               ) FROM installments_features""",

        "INSTAL_DELAY_MAX max value":
            "SELECT MAX(INSTAL_DELAY_MAX) FROM installments_features",

        "INSTAL_EVER_LATE count":
            "SELECT SUM(INSTAL_EVER_LATE) FROM installments_features",

        "INSTAL_PAYMENT_RATIO_MEAN > 2 (suspicious)":
            """SELECT COUNT(*) FROM installments_features
               WHERE INSTAL_PAYMENT_RATIO_MEAN > 2""",

        "INSTAL_DELAY_TREND nulls":
            """SELECT COUNT(*) FROM installments_features
               WHERE INSTAL_DELAY_TREND IS NULL""",
    }

    print("Sanity checks on installments_features\n")
    print(f"{'Check':<50} {'Result':>15}")
    print("─" * 68)
    for label, query in checks.items():
        result = con.execute(query).fetchone()[0]
        try:
            print(f"{label:<55} {result:>15,}  ✓")
        except (ValueError, TypeError):
            print(f"{label:<55} {str(result):>15}  ✓")
            
build_installments_features(con)
sanity_check_installments_features(con)

Step 1/4 — computing base metrics...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Step 1/4 — done.
Step 2/4 — counts and late stats...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Step 2/4 — done.
Step 3/4 — delay and amount stats...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Step 3/4 — done.
Step 4/4 — recency and trend...
Step 4/4 — done.
Joining parts...
Done. Table 'installments_features' created.
Sanity checks on installments_features

Check                                                       Result
────────────────────────────────────────────────────────────────────
Rows in installments_features                                   339,578  ✓
Clients in app_train with installments data                     291,635  ✓
Clients in app_train WITHOUT installments data                   15,876  ✓
INSTAL_COUNT mean                                                  40.1  ✓
INSTAL_LATE_RATIO range [min, max]                            0.0 — 1.0  ✓
INSTAL_DELAY_MAX max value                                      2,884.0  ✓
INSTAL_EVER_LATE count                                          179,845  ✓
INSTAL_PAYMENT_RATIO_MEAN > 2 (suspicious)                        3,925  ✓
INSTAL_DELAY_TREND nulls                                        133,350  ✓


### 3f — credit_card_balance

**What it captures**: monthly credit card statement history — balance, credit limit, payments, drawings, and DPD per statement.

**Key signals**: utilization rate (balance / limit), ATM drawing ratio (proxy for liquidity stress), payment ratio, over-limit months.

**ATM drawing ratio**: clients who frequently withdraw cash from their credit card (rather than using it for direct purchases) tend to be under liquidity pressure — a behavioural signal not captured by balance alone.

**Utilization trend**: more predictive than absolute utilization — a client moving from 30% to 80% utilization is riskier than one stable at 80%.

**Most predictive features**: `CC_UTILIZATION_MEAN`, `CC_EVER_LATE`, `CC_UTILIZATION_TREND`, `CC_EVER_OVER_LIMIT`.

In [12]:
def build_cc_features(con):
    """
    Block 3f — Aggregations from credit_card table.
    Split into parts to avoid memory issues (3.8M rows).

    Features created:
      - Balance and credit limit statistics
      - Utilization rate (balance / limit)
      - Payment behaviour (ratio, min payment)
      - Drawing statistics (ATM vs POS)
      - DPD statistics
      - Recency features
    """

    # Step 1 — base table
    print("Step 1/4 — computing base metrics...")
    con.execute("""
        CREATE OR REPLACE TABLE cc_base AS
        SELECT
            SK_ID_CURR,
            SK_ID_PREV,
            MONTHS_BALANCE,
            AMT_BALANCE,
            AMT_CREDIT_LIMIT_ACTUAL,
            AMT_PAYMENT_CURRENT,
            AMT_PAYMENT_TOTAL_CURRENT,
            AMT_RECEIVABLE_PRINCIPAL,
            AMT_RECIVABLE,
            AMT_TOTAL_RECEIVABLE,
            AMT_DRAWINGS_ATM_CURRENT,
            AMT_DRAWINGS_CURRENT,
            CNT_DRAWINGS_ATM_CURRENT,
            CNT_DRAWINGS_CURRENT,
            SK_DPD,
            SK_DPD_DEF,
            -- Utilization rate
            AMT_BALANCE / NULLIF(AMT_CREDIT_LIMIT_ACTUAL, 0) AS UTILIZATION,
            -- Payment ratio
            AMT_PAYMENT_CURRENT / NULLIF(AMT_TOTAL_RECEIVABLE, 0) AS PAYMENT_RATIO,
            -- ATM drawing ratio
            AMT_DRAWINGS_ATM_CURRENT / NULLIF(AMT_DRAWINGS_CURRENT, 0) AS ATM_RATIO
        FROM credit_card
        WHERE AMT_CREDIT_LIMIT_ACTUAL > 0
    """)
    print("Step 1/4 — done.")

    # Step 2 — balance and utilization stats
    print("Step 2/4 — balance and utilization...")
    con.execute("""
        CREATE OR REPLACE TABLE cc_part1 AS
        SELECT
            SK_ID_CURR,
            COUNT(*)                                        AS CC_MONTHS_COUNT,
            COUNT(DISTINCT SK_ID_PREV)                      AS CC_CONTRACT_COUNT,
            ROUND(AVG(AMT_BALANCE), 2)                     AS CC_BALANCE_MEAN,
            MAX(AMT_BALANCE)                               AS CC_BALANCE_MAX,
            ROUND(AVG(AMT_CREDIT_LIMIT_ACTUAL), 2)         AS CC_LIMIT_MEAN,
            MAX(AMT_CREDIT_LIMIT_ACTUAL)                   AS CC_LIMIT_MAX,
            ROUND(AVG(UTILIZATION), 4)                     AS CC_UTILIZATION_MEAN,
            MAX(UTILIZATION)                               AS CC_UTILIZATION_MAX,
            SUM(CASE WHEN UTILIZATION > 1
                     THEN 1 ELSE 0 END)                    AS CC_OVER_LIMIT_COUNT,
            MAX(CASE WHEN UTILIZATION > 1
                     THEN 1 ELSE 0 END)                    AS CC_EVER_OVER_LIMIT
        FROM cc_base
        GROUP BY SK_ID_CURR
    """)
    print("Step 2/4 — done.")

    # Step 3 — payment and drawing stats
    print("Step 3/4 — payment and drawing stats...")
    con.execute("""
        CREATE OR REPLACE TABLE cc_part2 AS
        SELECT
            SK_ID_CURR,
            ROUND(AVG(PAYMENT_RATIO), 4)                   AS CC_PAYMENT_RATIO_MEAN,
            MIN(PAYMENT_RATIO)                             AS CC_PAYMENT_RATIO_MIN,
            ROUND(AVG(AMT_PAYMENT_CURRENT), 2)             AS CC_PAYMENT_MEAN,
            ROUND(AVG(AMT_DRAWINGS_CURRENT), 2)            AS CC_DRAWINGS_MEAN,
            ROUND(AVG(AMT_DRAWINGS_ATM_CURRENT), 2)        AS CC_ATM_DRAWINGS_MEAN,
            ROUND(AVG(ATM_RATIO), 4)                       AS CC_ATM_RATIO_MEAN,
            ROUND(AVG(CNT_DRAWINGS_CURRENT), 2)            AS CC_CNT_DRAWINGS_MEAN,
            ROUND(AVG(CNT_DRAWINGS_ATM_CURRENT), 2)        AS CC_CNT_ATM_DRAWINGS_MEAN,
            -- Receivable stats
            ROUND(AVG(AMT_TOTAL_RECEIVABLE), 2)            AS CC_RECEIVABLE_MEAN,
            MAX(AMT_TOTAL_RECEIVABLE)                      AS CC_RECEIVABLE_MAX
        FROM cc_base
        GROUP BY SK_ID_CURR
    """)
    print("Step 3/4 — done.")

    # Step 4 — DPD and recency
    print("Step 4/4 — DPD and recency...")
    con.execute("""
        CREATE OR REPLACE TABLE cc_part3 AS
        SELECT
            SK_ID_CURR,
            MAX(SK_DPD)                                     AS CC_DPD_MAX,
            ROUND(AVG(SK_DPD), 4)                          AS CC_DPD_MEAN,
            MAX(SK_DPD_DEF)                                 AS CC_DPD_DEF_MAX,
            SUM(CASE WHEN SK_DPD > 0 THEN 1 ELSE 0 END)    AS CC_DPD_COUNT,
            MAX(CASE WHEN SK_DPD > 0 THEN 1 ELSE 0 END)    AS CC_EVER_LATE,
            -- Recency — last 12 months
            ROUND(AVG(CASE WHEN MONTHS_BALANCE >= -12
                          THEN UTILIZATION END), 4)        AS CC_UTILIZATION_MEAN_12M,
            ROUND(AVG(CASE WHEN MONTHS_BALANCE >= -12
                          THEN SK_DPD END), 4)             AS CC_DPD_MEAN_12M,
            MAX(CASE WHEN MONTHS_BALANCE >= -12
                     THEN SK_DPD END)                      AS CC_DPD_MAX_12M,
            -- Trend
            ROUND(
                AVG(CASE WHEN MONTHS_BALANCE >= -6  THEN UTILIZATION END) -
                AVG(CASE WHEN MONTHS_BALANCE <  -6  THEN UTILIZATION END)
            , 4)                                           AS CC_UTILIZATION_TREND
        FROM cc_base
        GROUP BY SK_ID_CURR
    """)
    print("Step 4/4 — done.")

    # Final join
    print("Joining parts...")
    con.execute("""
        CREATE OR REPLACE TABLE cc_features AS
        SELECT
            p1.*,
            p2.CC_PAYMENT_RATIO_MEAN,
            p2.CC_PAYMENT_RATIO_MIN,
            p2.CC_PAYMENT_MEAN,
            p2.CC_DRAWINGS_MEAN,
            p2.CC_ATM_DRAWINGS_MEAN,
            p2.CC_ATM_RATIO_MEAN,
            p2.CC_CNT_DRAWINGS_MEAN,
            p2.CC_CNT_ATM_DRAWINGS_MEAN,
            p2.CC_RECEIVABLE_MEAN,
            p2.CC_RECEIVABLE_MAX,
            p3.CC_DPD_MAX,
            p3.CC_DPD_MEAN,
            p3.CC_DPD_DEF_MAX,
            p3.CC_DPD_COUNT,
            p3.CC_EVER_LATE,
            p3.CC_UTILIZATION_MEAN_12M,
            p3.CC_DPD_MEAN_12M,
            p3.CC_DPD_MAX_12M,
            p3.CC_UTILIZATION_TREND
        FROM cc_part1 p1
        LEFT JOIN cc_part2 p2 USING (SK_ID_CURR)
        LEFT JOIN cc_part3 p3 USING (SK_ID_CURR)
    """)

    # Cleanup
    for t in ['cc_base', 'cc_part1', 'cc_part2', 'cc_part3']:
        con.execute(f"DROP TABLE IF EXISTS {t}")
    print("Done. Table 'cc_features' created.")


def sanity_check_cc_features(con):
    checks = {
        "Rows in cc_features":
            "SELECT COUNT(*) FROM cc_features",

        "Clients in app_train with cc data":
            """SELECT COUNT(DISTINCT c.SK_ID_CURR)
               FROM cc_features c
               JOIN app_train a USING (SK_ID_CURR)""",

        "Clients in app_train WITHOUT cc data":
            """SELECT COUNT(*)
               FROM app_train a
               LEFT JOIN cc_features c USING (SK_ID_CURR)
               WHERE c.SK_ID_CURR IS NULL""",

        "CC_UTILIZATION_MEAN range [min, max]":
            """SELECT CONCAT(
                ROUND(MIN(CC_UTILIZATION_MEAN), 3), ' — ',
                ROUND(MAX(CC_UTILIZATION_MEAN), 3)
               ) FROM cc_features""",

        "CC_EVER_OVER_LIMIT count":
            "SELECT SUM(CC_EVER_OVER_LIMIT) FROM cc_features",

        "CC_DPD_MAX max value":
            "SELECT MAX(CC_DPD_MAX) FROM cc_features",

        "CC_EVER_LATE count":
            "SELECT SUM(CC_EVER_LATE) FROM cc_features",

        "CC_UTILIZATION_TREND nulls":
            """SELECT COUNT(*) FROM cc_features
               WHERE CC_UTILIZATION_TREND IS NULL""",

        "CC_PAYMENT_RATIO_MEAN > 3 (suspicious)":
            """SELECT COUNT(*) FROM cc_features
               WHERE CC_PAYMENT_RATIO_MEAN > 3""",
    }

    print("Sanity checks on cc_features\n")
    print(f"{'Check':<50} {'Result':>15}")
    print("─" * 68)
    for label, query in checks.items():
        result = con.execute(query).fetchone()[0]
        try:
            print(f"{label:<50} {result:>15,}  ✓")
        except (ValueError, TypeError):
            print(f"{label:<50} {str(result):>15}  ✓")


build_cc_features(con)
sanity_check_cc_features(con)

Step 1/4 — computing base metrics...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Step 1/4 — done.
Step 2/4 — balance and utilization...
Step 2/4 — done.
Step 3/4 — payment and drawing stats...
Step 3/4 — done.
Step 4/4 — DPD and recency...
Step 4/4 — done.
Joining parts...
Done. Table 'cc_features' created.
Sanity checks on cc_features

Check                                                       Result
────────────────────────────────────────────────────────────────────
Rows in cc_features                                        102,445  ✓
Clients in app_train with cc data                           86,036  ✓
Clients in app_train WITHOUT cc data                       221,475  ✓
CC_UTILIZATION_MEAN range [min, max]                -0.085 — 2.139  ✓
CC_EVER_OVER_LIMIT count                                    43,646  ✓
CC_DPD_MAX max value                                         3,260  ✓
CC_EVER_LATE count                                          20,571  ✓
CC_UTILIZATION_TREND nulls                                  29,592  ✓
CC_PAYMENT_RATIO_MEAN > 3 (suspicious)        

### 3g — Final Join

Joins `app_train_features` with all six secondary feature tables into a single flat table `app_train_final`.

**Join strategy**: `LEFT JOIN` on `SK_ID_CURR` throughout — clients with no history in a secondary table receive `NULL`, not a dropped row. This preserve all 307,511 clients.

**Memory strategy**: the join is split into three sequential steps, each materializing an intermediate table to disk before proceeding.  
This avoids loading all feature tables simultaneously into RAM.

In [13]:
# Materialize app_train_features as a table first
print("Materializing app_train_features...")
con.execute("""
    CREATE OR REPLACE TABLE app_train_features_tbl AS
    SELECT * FROM app_train_features
""")
print("Done.")

Materializing app_train_features...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Done.


In [14]:
print("Step 1/3 — joining bureau features...")
con.execute("""
    CREATE OR REPLACE TABLE app_train_temp1 AS
    SELECT
        a.*,
        b.BUREAU_LOAN_COUNT,
        b.BUREAU_ACTIVE_COUNT,
        b.BUREAU_CLOSED_COUNT,
        b.BUREAU_ACTIVE_RATIO,
        b.BUREAU_CREDIT_SUM,
        b.BUREAU_CREDIT_MEAN,
        b.BUREAU_CREDIT_MAX,
        b.BUREAU_DEBT_SUM,
        b.BUREAU_DEBT_MEAN,
        b.BUREAU_OVERDUE_SUM,
        b.BUREAU_OVERDUE_MEAN,
        b.BUREAU_MAX_DAY_OVERDUE,
        b.BUREAU_MEAN_DAY_OVERDUE,
        b.BUREAU_OVERDUE_COUNT,
        b.BUREAU_DEBT_CREDIT_RATIO,
        b.BUREAU_DAYS_CREDIT_MIN,
        b.BUREAU_DAYS_CREDIT_MEAN,
        b.BUREAU_ENDDATE_MAX,
        b.BUREAU_DAYS_UPDATE_MEAN,
        b.BUREAU_PROLONG_SUM,
        b.BUREAU_PROLONG_MEAN,
        b.BUREAU_CONSUMER_COUNT,
        b.BUREAU_CARD_COUNT,
        b.BUREAU_MORTGAGE_COUNT,
        b.BUREAU_CAR_COUNT,
        b.BUREAU_HAS_BAD_DEBT,
        b.BUREAU_EVER_OVERDUE,
        bb.BBAL_MONTHS_COUNT,
        bb.BBAL_UNIQUE_MONTHS,
        bb.BBAL_STATUS_C_RATIO,
        bb.BBAL_STATUS_0_RATIO,
        bb.BBAL_DPD_RATIO,
        bb.BBAL_DPD_MAX,
        bb.BBAL_DPD_MEAN,
        bb.BBAL_DPD_COUNT,
        bb.BBAL_EVER_LATE,
        bb.BBAL_EVER_LATE_3M,
        bb.BBAL_DPD_MEAN_12M,
        bb.BBAL_DPD_MAX_12M,
        bb.BBAL_DPD_MEAN_6M,
        bb.BBAL_DPD_TREND
    FROM app_train_features_tbl        a
    LEFT JOIN bureau_features      b  USING (SK_ID_CURR)
    LEFT JOIN bureau_bal_features  bb USING (SK_ID_CURR)
""")
print("Step 1/3 — done.")

Step 1/3 — joining bureau features...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Step 1/3 — done.


In [13]:
print("Step 2/3 — joining prev_app and pos features...")
con.execute("""
    CREATE OR REPLACE TABLE app_train_temp2 AS
    SELECT
        t.*,
        p.PREV_APP_COUNT,
        p.PREV_APPROVED_COUNT,
        p.PREV_REFUSED_COUNT,
        p.PREV_CANCELED_COUNT,
        p.PREV_APPROVAL_RATE,
        p.PREV_REFUSAL_RATE,
        p.PREV_AMT_CREDIT_MEAN,
        p.PREV_AMT_CREDIT_MAX,
        p.PREV_AMT_APPLICATION_MEAN,
        p.PREV_CREDIT_APPLICATION_RATIO,
        p.PREV_AMT_ANNUITY_MEAN,
        p.PREV_AMT_ANNUITY_MAX,
        p.PREV_DOWN_PAYMENT_MEAN,
        p.PREV_DOWN_PAYMENT_RATE_MEAN,
        p.PREV_CNT_PAYMENT_MEAN,
        p.PREV_CNT_PAYMENT_MAX,
        p.PREV_DAYS_DECISION_LAST,
        p.PREV_DAYS_DECISION_MEAN,
        p.PREV_CONSUMER_COUNT,
        p.PREV_CASH_COUNT,
        p.PREV_REVOLVING_COUNT,
        p.PREV_LAST_STATUS,
        p.PREV_LAST_CREDIT,
        p.PREV_LAST_CNT_PAYMENT,
        p.PREV_EVER_REFUSED,
        p.PREV_EVER_HIGH_YIELD,
        p.PREV_NFLAG_INSURED_MEAN,
        p.PREV_INTEREST_RATE_MEAN,
        p.PREV_AMT_CREDIT_MEAN_12M,
        p.PREV_APP_COUNT_12M,
        p.PREV_APPROVAL_RATE_12M,
        p.PREV_CASH_CREDIT_MEAN,
        p.PREV_REVOLVING_CREDIT_MEAN,
        p.PREV_CONSUMER_CREDIT_MEAN,
        p.PREV_CASH_ANNUITY_MEAN,
        p.PREV_REVOLVING_ANNUITY_MEAN,
        pos.POS_MONTHS_COUNT,
        pos.POS_CONTRACT_COUNT,
        pos.POS_COMPLETED_RATIO,
        pos.POS_EVER_COMPLETED,
        pos.POS_ACTIVE_COUNT,
        pos.POS_DPD_MAX,
        pos.POS_DPD_MEAN,
        pos.POS_DPD_DEF_MAX,
        pos.POS_DPD_DEF_MEAN,
        pos.POS_DPD_COUNT,
        pos.POS_EVER_LATE,
        pos.POS_EVER_LATE_30,
        pos.POS_EVER_LATE_60,
        pos.POS_DPD_MEAN_12M,
        pos.POS_DPD_MAX_12M,
        pos.POS_DPD_MEAN_6M,
        pos.POS_DPD_COUNT_12M,
        pos.POS_DPD_TREND,
        pos.POS_CNT_INSTALMENT_MEAN,
        pos.POS_CNT_INSTALMENT_FUTURE_MEAN
    FROM app_train_temp1           t
    LEFT JOIN prev_app_features    p   USING (SK_ID_CURR)
    LEFT JOIN pos_features         pos USING (SK_ID_CURR)
""")
con.execute("DROP TABLE IF EXISTS app_train_temp1")
print("Step 2/3 — done.")

Step 2/3 — joining prev_app and pos features...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

OutOfMemoryException: Out of Memory Error: Allocation failure

In [ ]:
print("Step 3/3 — joining installments and cc features...")
con.execute("""
    CREATE OR REPLACE TABLE app_train_final AS
    SELECT
        t.*,
        i.INSTAL_COUNT,
        i.INSTAL_CONTRACT_COUNT,
        i.INSTAL_LATE_COUNT,
        i.INSTAL_LATE_RATIO,
        i.INSTAL_EVER_LATE,
        i.INSTAL_EVER_LATE_30,
        i.INSTAL_EARLY_COUNT,
        i.INSTAL_DELAY_MEAN,
        i.INSTAL_DELAY_MAX,
        i.INSTAL_DELAY_MEAN_LATE,
        i.INSTAL_EARLY_DAYS_MEAN,
        i.INSTAL_PAYMENT_RATIO_MEAN,
        i.INSTAL_PAYMENT_RATIO_MIN,
        i.INSTAL_UNDERPAID_COUNT,
        i.INSTAL_EVER_UNDERPAID,
        i.INSTAL_AMT_MEAN,
        i.INSTAL_PAYMENT_MEAN,
        i.INSTAL_AMT_DIFF_SUM,
        i.INSTAL_DELAY_MEAN_12M,
        i.INSTAL_DELAY_MAX_12M,
        i.INSTAL_DELAY_MEAN_6M,
        i.INSTAL_LATE_RATIO_12M,
        i.INSTAL_DELAY_TREND,
        cc.CC_MONTHS_COUNT,
        cc.CC_CONTRACT_COUNT,
        cc.CC_BALANCE_MEAN,
        cc.CC_BALANCE_MAX,
        cc.CC_LIMIT_MEAN,
        cc.CC_LIMIT_MAX,
        cc.CC_UTILIZATION_MEAN,
        cc.CC_UTILIZATION_MAX,
        cc.CC_OVER_LIMIT_COUNT,
        cc.CC_EVER_OVER_LIMIT,
        cc.CC_PAYMENT_RATIO_MEAN,
        cc.CC_PAYMENT_RATIO_MIN,
        cc.CC_PAYMENT_MEAN,
        cc.CC_DRAWINGS_MEAN,
        cc.CC_ATM_DRAWINGS_MEAN,
        cc.CC_ATM_RATIO_MEAN,
        cc.CC_CNT_DRAWINGS_MEAN,
        cc.CC_CNT_ATM_DRAWINGS_MEAN,
        cc.CC_RECEIVABLE_MEAN,
        cc.CC_RECEIVABLE_MAX,
        cc.CC_DPD_MAX,
        cc.CC_DPD_MEAN,
        cc.CC_DPD_DEF_MAX,
        cc.CC_DPD_COUNT,
        cc.CC_EVER_LATE,
        cc.CC_UTILIZATION_MEAN_12M,
        cc.CC_DPD_MEAN_12M,
        cc.CC_DPD_MAX_12M,
        cc.CC_UTILIZATION_TREND
    FROM app_train_temp2            t
    LEFT JOIN installments_features i  USING (SK_ID_CURR)
    LEFT JOIN cc_features           cc USING (SK_ID_CURR)
""")
con.execute("DROP TABLE IF EXISTS app_train_temp2")
print("Step 3/3 — done.")

Step 3/3 — joining installments and cc features...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Step 3/3 — done.


In [ ]:
def sanity_check_final_table(con):
    checks = {
        "Rows in app_train_final":
            "SELECT COUNT(*) FROM app_train_final",

        "Total features (columns)":
            """SELECT COUNT(*) FROM information_schema.columns
               WHERE table_name = 'app_train_final'""",

        "Rows match app_train":
            """SELECT COUNT(*) = (SELECT COUNT(*) FROM app_train)
               FROM app_train_final""",

        "TARGET null count":
            "SELECT COUNT(*) FROM app_train_final WHERE TARGET IS NULL",

        "SK_ID_CURR duplicates":
            """SELECT COUNT(*) FROM (
                SELECT SK_ID_CURR FROM app_train_final
                GROUP BY SK_ID_CURR HAVING COUNT(*) > 1
               )""",

        "Clients with no bureau data (NULL)":
            """SELECT COUNT(*) FROM app_train_final
               WHERE BUREAU_LOAN_COUNT IS NULL""",

        "Clients with no installments data (NULL)":
            """SELECT COUNT(*) FROM app_train_final
               WHERE INSTAL_COUNT IS NULL""",

        "Clients with no cc data (NULL)":
            """SELECT COUNT(*) FROM app_train_final
               WHERE CC_MONTHS_COUNT IS NULL""",

        "Clients with no prev_app data (NULL)":
            """SELECT COUNT(*) FROM app_train_final
               WHERE PREV_APP_COUNT IS NULL""",
    }

    print("Sanity checks on app_train_final\n")
    print(f"{'Check':<45} {'Result':>15}")
    print("─" * 63)
    for label, query in checks.items():
        result = con.execute(query).fetchone()[0]
        try:
            print(f"{label:<45} {result:>15,}  ✓")
        except (ValueError, TypeError):
            print(f"{label:<45} {str(result):>15}  ✓")

sanity_check_final_table(con)

Sanity checks on app_train_final

Check                                                  Result
───────────────────────────────────────────────────────────────
Rows in app_train_final                               307,511  ✓
Total features (columns)                                  245  ✓
Rows match app_train                                        1  ✓
TARGET null count                                           0  ✓
SK_ID_CURR duplicates                                       0  ✓
Clients with no bureau data (NULL)                     44,020  ✓
Clients with no installments data (NULL)               15,876  ✓
Clients with no cc data (NULL)                        221,475  ✓
Clients with no prev_app data (NULL)                   16,601  ✓


## Block 4 — Cross-Table Features

Single-table features capture behaviour within one product type.  
Cross-table features capture **relationships and consistency across tables** — signals that no single table can express alone.

**Eight groups of features created:**

**1. Current loan vs historical behaviour**  
Compares the current application amounts against what HC approved in the past.  
A client requesting significantly more than their historical average is a risk signal (`CREDIT_VS_PREV_RATIO`, `ANNUITY_VS_PREV_RATIO`).

**2. Total debt exposure**  
Combines the current loan with all external bureau debt to compute true total exposure relative to income.  
`INCOME_TO_TOTAL_DEBT_RATIO` is a more complete debt burden measure than `CREDIT_INCOME_RATIO` alone.

**3. Overdue signals vs income**  
Scales external overdue amounts by income and current loan size — a €500 overdue on a €5,000 income is very different from €500 on €50,000.

**4. Behavioural consistency across HC products**  
Aggregates late payment signals across POS, installments and credit card:  
- `EVER_LATE_ANY_PRODUCT` — was late on at least one HC product  
- `EVER_LATE_ALL_PRODUCTS` — was late on every HC product (severe signal)  
- `N_PRODUCTS_EVER_LATE` — count of product types with late history

**5. Credit history depth**  
Total contracts and total months of payment history across all HC products — a proxy for how well-known the client is to HC.

**6. EXT_SOURCE vs internal behaviour**  
Adjusts external credit scores downward when internal HC behaviour contradicts them.  
A high external score combined with a high HC refusal rate signals inconsistency (`EXT_SCORE_ADJ_REFUSAL`, `EXT_SCORE_VS_INTERNAL_DPD`).

**7. Payment behaviour consistency**  
Averages late payment signals across products and combines worsening trends from installments, POS and credit card into a single `COMBINED_WORSENING_TREND`.  
If all three trend signals are positive, the risk is compounding.

**8. Financial stress indicators**  
`FINANCIAL_STRESS_SCORE` combines credit card utilization and annuity burden.  
`OVER_LIMIT_AND_LATE` flags clients who are simultaneously over their credit limit and paying installments late — the most severe stress combination.

In [ ]:
def build_cross_table_features(con):
    """
    Block 4 — Cross-table features.
    Combines signals from multiple tables into a new set of features
    that capture relationships no single table can express alone.

    Built on top of app_train_final — adds a new table
    'app_train_final_v2' with all original features + cross-table ones.
    """

    print("Building cross-table features...")
    con.execute("""
        CREATE OR REPLACE TABLE app_train_final_v2 AS
        SELECT
            *,

            -- ── 1. Current loan vs historical behaviour ───────────────

            -- Is the current loan larger than what HC approved in the past?
            ROUND(
                AMT_CREDIT / NULLIF(PREV_AMT_CREDIT_MEAN, 0)
            , 4)                                AS CREDIT_VS_PREV_RATIO,

            -- Is the current annuity burden higher than historical?
            ROUND(
                AMT_ANNUITY / NULLIF(PREV_AMT_ANNUITY_MEAN, 0)
            , 4)                                AS ANNUITY_VS_PREV_RATIO,

            -- Is the client requesting more than they usually get approved?
            ROUND(
                AMT_CREDIT / NULLIF(PREV_AMT_APPLICATION_MEAN, 0)
            , 4)                                AS CREDIT_VS_PREV_APPLICATION_RATIO,

            -- ── 2. Total debt exposure ────────────────────────────────

            -- Total debt: current loan + all external bureau debt
            AMT_CREDIT + COALESCE(BUREAU_DEBT_SUM, 0)
                                                AS TOTAL_DEBT,

            -- Income vs total debt exposure (internal + external)
            ROUND(
                AMT_INCOME_TOTAL /
                NULLIF(AMT_CREDIT + COALESCE(BUREAU_DEBT_SUM, 0), 0)
            , 4)                                AS INCOME_TO_TOTAL_DEBT_RATIO,

            -- Total annuity burden: current + estimated bureau annuities
            ROUND(
                AMT_ANNUITY /
                NULLIF(AMT_INCOME_TOTAL - COALESCE(BUREAU_DEBT_MEAN, 0), 0)
            , 4)                                AS ANNUITY_TO_RESIDUAL_INCOME,

            -- ── 3. Overdue signals vs income ──────────────────────────

            -- External overdue relative to income
            ROUND(
                COALESCE(BUREAU_OVERDUE_MEAN, 0) /
                NULLIF(AMT_INCOME_TOTAL, 0)
            , 6)                                AS OVERDUE_TO_INCOME_RATIO,

            -- External overdue relative to current loan
            ROUND(
                COALESCE(BUREAU_OVERDUE_SUM, 0) /
                NULLIF(AMT_CREDIT, 0)
            , 4)                                AS OVERDUE_TO_CREDIT_RATIO,

            -- ── 4. Behavioural consistency across tables ──────────────

            -- DPD signal aggregated across all HC internal tables
            -- High = client has been late across multiple product types
            (COALESCE(POS_DPD_MAX,   0) +
             COALESCE(INSTAL_DELAY_MAX, 0) +
             COALESCE(CC_DPD_MAX,    0))        AS TOTAL_DPD_SCORE,

            -- Ever late across ANY HC internal product
            CASE WHEN COALESCE(POS_EVER_LATE,   0) = 1
                   OR COALESCE(INSTAL_EVER_LATE, 0) = 1
                   OR COALESCE(CC_EVER_LATE,     0) = 1
                 THEN 1 ELSE 0
            END                                 AS EVER_LATE_ANY_PRODUCT,

            -- Ever late across ALL HC internal products (more severe signal)
            CASE WHEN COALESCE(POS_EVER_LATE,   0) = 1
                  AND COALESCE(INSTAL_EVER_LATE, 0) = 1
                  AND COALESCE(CC_EVER_LATE,     0) = 1
                 THEN 1 ELSE 0
            END                                 AS EVER_LATE_ALL_PRODUCTS,

            -- Number of product types where client was ever late
            (COALESCE(POS_EVER_LATE,   0) +
             COALESCE(INSTAL_EVER_LATE, 0) +
             COALESCE(CC_EVER_LATE,     0) +
             COALESCE(BBAL_EVER_LATE,   0))     AS N_PRODUCTS_EVER_LATE,

            -- ── 5. Credit history depth ───────────────────────────────

            -- Total number of contracts across all HC products
            COALESCE(PREV_APP_COUNT,      0) +
            COALESCE(POS_CONTRACT_COUNT,  0) +
            COALESCE(CC_CONTRACT_COUNT,   0)    AS TOTAL_HC_CONTRACTS,

            -- Total months of payment history across all products
            COALESCE(BBAL_MONTHS_COUNT,   0) +
            COALESCE(POS_MONTHS_COUNT,    0) +
            COALESCE(INSTAL_COUNT,        0) +
            COALESCE(CC_MONTHS_COUNT,     0)    AS TOTAL_HISTORY_MONTHS,

            -- ── 6. EXT_SOURCE vs internal behaviour ───────────────────

            -- External score vs internal DPD (disagreement = uncertainty)
            ROUND(
                EXT_SOURCE_MEAN *
                (1 - LEAST(COALESCE(BBAL_DPD_RATIO, 0), 1))
            , 4)                                AS EXT_SCORE_VS_INTERNAL_DPD,

            -- External score weighted by refusal history
            ROUND(
                EXT_SOURCE_MEAN *
                (1 - COALESCE(PREV_REFUSAL_RATE, 0))
            , 4)                                AS EXT_SCORE_ADJ_REFUSAL,

            -- ── 7. Payment behaviour consistency ─────────────────────

            -- Average late ratio across installments and POS
            ROUND(
                (COALESCE(INSTAL_LATE_RATIO, 0) +
                 COALESCE(POS_DPD_MEAN,      0) / 30.0)
                / 2.0
            , 4)                                AS AVG_LATE_RATIO_HC,

            -- Worsening trend across products (positive = worsening)
            ROUND(
                COALESCE(INSTAL_DELAY_TREND, 0) +
                COALESCE(POS_DPD_TREND,      0) +
                COALESCE(CC_UTILIZATION_TREND, 0)
            , 4)                                AS COMBINED_WORSENING_TREND,

            -- ── 8. Financial stress indicators ───────────────────────

            -- CC utilization + annuity burden combined stress score
            ROUND(
                COALESCE(CC_UTILIZATION_MEAN, 0) +
                COALESCE(ANNUITY_INCOME_RATIO, 0)
            , 4)                                AS FINANCIAL_STRESS_SCORE,

            -- Client is over limit AND paying late (severe stress)
            CASE WHEN COALESCE(CC_EVER_OVER_LIMIT, 0) = 1
                  AND COALESCE(INSTAL_EVER_LATE,    0) = 1
                 THEN 1 ELSE 0
            END                                 AS OVER_LIMIT_AND_LATE

        FROM app_train_final
    """)
    print("Done. Table 'app_train_final_v2' created.")


def sanity_check_cross_table_features(con):
    checks = {
        "Rows in app_train_final_v2":
            "SELECT COUNT(*) FROM app_train_final_v2",

        "Total columns":
            """SELECT COUNT(*) FROM information_schema.columns
               WHERE table_name = 'app_train_final_v2'""",

        "TOTAL_DEBT max":
            "SELECT ROUND(MAX(TOTAL_DEBT), 0) FROM app_train_final_v2",

        "EVER_LATE_ANY_PRODUCT count":
            "SELECT SUM(EVER_LATE_ANY_PRODUCT) FROM app_train_final_v2",

        "EVER_LATE_ALL_PRODUCTS count":
            "SELECT SUM(EVER_LATE_ALL_PRODUCTS) FROM app_train_final_v2",

        "N_PRODUCTS_EVER_LATE distribution [min, max]":
            """SELECT CONCAT(
                MIN(N_PRODUCTS_EVER_LATE), ' — ',
                MAX(N_PRODUCTS_EVER_LATE)
               ) FROM app_train_final_v2""",

        "INCOME_TO_TOTAL_DEBT_RATIO nulls":
            """SELECT COUNT(*) FROM app_train_final_v2
               WHERE INCOME_TO_TOTAL_DEBT_RATIO IS NULL""",

        "OVER_LIMIT_AND_LATE count":
            "SELECT SUM(OVER_LIMIT_AND_LATE) FROM app_train_final_v2",

        "COMBINED_WORSENING_TREND nulls":
            """SELECT COUNT(*) FROM app_train_final_v2
               WHERE COMBINED_WORSENING_TREND IS NULL""",
    }

    print("Sanity checks on app_train_final_v2\n")
    print(f"{'Check':<50} {'Result':>15}")
    print("─" * 68)
    for label, query in checks.items():
        result = con.execute(query).fetchone()[0]
        try:
            print(f"{label:<50} {result:>15,}  ✓")
        except (ValueError, TypeError):
            print(f"{label:<50} {str(result):>15}  ✓")


build_cross_table_features(con)
sanity_check_cross_table_features(con)

Building cross-table features...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Done. Table 'app_train_final_v2' created.
Sanity checks on app_train_final_v2

Check                                                       Result
────────────────────────────────────────────────────────────────────
Rows in app_train_final_v2                                 307,511  ✓
Total columns                                                  265  ✓
TOTAL_DEBT max                                       335,443,331.0  ✓
EVER_LATE_ANY_PRODUCT count                                155,775  ✓
EVER_LATE_ALL_PRODUCTS count                                 4,748  ✓
N_PRODUCTS_EVER_LATE distribution [min, max]                 0 — 4  ✓
INCOME_TO_TOTAL_DEBT_RATIO nulls                                 0  ✓
OVER_LIMIT_AND_LATE count                                   30,759  ✓
COMBINED_WORSENING_TREND nulls                                   0  ✓


## Block 5 — Categorical Encoding

Converts remaining string columns into numeric representations
that the model can consume directly.

**Three encoding strategies applied:**

**1. One-hot encoding — `PREV_LAST_STATUS`**  
Low cardinality (3 categories: Approved, Refused, Canceled), no leakage risk.  
Encoded directly into `app_train_final_v3` as three binary columns:  
`PREV_LAST_APPROVED`, `PREV_LAST_REFUSED`, `PREV_LAST_CANCELED`.

**2. Target encoding — `OCCUPATION_TYPE`, `ORGANIZATION_TYPE`**  
High cardinality (18 and 58 categories respectively) — one-hot would add  
too many sparse columns and hurt model performance.  
Target encoding replaces each category with the smoothed default rate  
of that category computed on the training fold only.  

**Why smoothing matters**: raw mean encoding overfits on rare categories.  
With `smooth=20`, a category with few observations is pulled strongly  
toward the global mean — preventing the model from learning noise.  

**Why inside the k-fold loop**: fitting the encoder on the full training set  
before splitting would let the validation fold's target values influence  
its own features — a form of data leakage. The `target_encode()` function  
defined here is called inside the loop at modelling time.

**3. Columns requiring no further encoding**  
All other categorical columns were already handled in Block 2:  
binary flags → 0/1, ordinal columns → integer scale,  
low-cardinality nominals → one-hot.Categorical Encoding

In [21]:
def build_categorical_encoding(con):
    """
    Block 5 — Categorical encoding.

    Handles three types:
      1. PREV_LAST_STATUS → one-hot (low cardinality, no leakage risk)
      2. Boolean-like strings → 0/1 integer
      3. OCCUPATION_TYPE, ORGANIZATION_TYPE → left as-is for target
         encoding inside the k-fold loop at modelling time

    Output: app_train_final_v3
    """
    con.execute("""
        CREATE OR REPLACE TABLE app_train_final_v3 AS
        SELECT
            *,

            -- ── PREV_LAST_STATUS one-hot ──────────────────────────────
            CASE WHEN PREV_LAST_STATUS = 'Approved'  THEN 1 ELSE 0 END
                                            AS PREV_LAST_APPROVED,
            CASE WHEN PREV_LAST_STATUS = 'Refused'   THEN 1 ELSE 0 END
                                            AS PREV_LAST_REFUSED,
            CASE WHEN PREV_LAST_STATUS = 'Canceled'  THEN 1 ELSE 0 END
                                            AS PREV_LAST_CANCELED

        FROM app_train_final_v2
    """)
    print("Done. Table 'app_train_final_v3' created.")


build_categorical_encoding(con)

OutOfMemoryException: Out of Memory Error: Allocation failure

In [22]:
# ── Verify remaining categorical columns ─────────────────────────────
# These are the columns LightGBM cannot handle as-is
# OCCUPATION_TYPE and ORGANIZATION_TYPE are kept as strings
# and will be target-encoded inside the k-fold loop

remaining_cats = con.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_name = 'app_train_final_v3'
      AND data_type = 'VARCHAR'
    ORDER BY column_name
""").fetchdf()

print("Remaining VARCHAR columns — require handling at model time:\n")
print(remaining_cats.to_string(index=False))

Remaining VARCHAR columns — require handling at model time:

      column_name data_type
  OCCUPATION_TYPE   VARCHAR
ORGANIZATION_TYPE   VARCHAR
 PREV_LAST_STATUS   VARCHAR


In [23]:
# ── Target encoding — to be used INSIDE the k-fold loop ──────────────
# This function is defined here but called inside the training loop.
# Fitting on train folds only prevents target leakage.

def target_encode(X_train, y_train, X_val, cols, smooth=20):
    """
    Smoothed target encoding — prevents overfitting on rare categories.

    Formula:
        encoded = (n_category * mean_category + smooth * global_mean)
                  / (n_category + smooth)

    smooth=20 means a category needs 20+ observations before its own
    mean dominates over the global mean. Rare categories are pulled
    strongly toward the global mean.

    Parameters:
        X_train  : pd.DataFrame — training fold features
        y_train  : pd.Series    — training fold target
        X_val    : pd.DataFrame — validation fold features
        cols     : list of str  — columns to encode
        smooth   : int          — smoothing factor

    Returns:
        X_train_enc, X_val_enc : encoded copies (originals unchanged)
    """
    X_train_enc = X_train.copy()
    X_val_enc   = X_val.copy()

    global_mean = y_train.mean()

    for col in cols:
        # Compute smoothed mean per category on train fold
        stats = (
            X_train_enc[[col]]
            .assign(target=y_train.values)
            .groupby(col)['target']
            .agg(['mean', 'count'])
        )
        stats['smoothed'] = (
            (stats['count'] * stats['mean'] + smooth * global_mean)
            / (stats['count'] + smooth)
        )
        mapping = stats['smoothed'].to_dict()

        # Apply to train fold
        X_train_enc[col] = (
            X_train_enc[col]
            .map(mapping)
            .fillna(global_mean)   # unseen categories → global mean
        )

        # Apply to val fold using train fold mapping only
        X_val_enc[col] = (
            X_val_enc[col]
            .map(mapping)
            .fillna(global_mean)
        )

    return X_train_enc, X_val_enc


# Columns to target-encode at model time
TARGET_ENCODE_COLS = ['OCCUPATION_TYPE', 'ORGANIZATION_TYPE']

print("Target encoding ready.")
print(f"Columns to encode inside k-fold: {TARGET_ENCODE_COLS}")

Target encoding ready.
Columns to encode inside k-fold: ['OCCUPATION_TYPE', 'ORGANIZATION_TYPE']


In [24]:
# ── Final check — what goes into the model ───────────────────────────

col_summary = con.execute("""
    SELECT
        data_type,
        COUNT(*) AS n_columns
    FROM information_schema.columns
    WHERE table_name = 'app_train_final_v3'
    GROUP BY data_type
    ORDER BY n_columns DESC
""").fetchdf()

print("app_train_final_v3 — column type summary:\n")
print(col_summary.to_string(index=False))
print(f"\nTotal columns: {col_summary['n_columns'].sum()}")

app_train_final_v3 — column type summary:

data_type  n_columns
   DOUBLE        146
   BIGINT         49
  INTEGER         46
  HUGEINT         24
  VARCHAR          3

Total columns: 268


## Test Set Pipeline

Applies the exact same feature engineering pipeline to `app_test`
to generate the test features required for the Kaggle submission.

**Key constraint**: all statistics and thresholds computed during
training are reused as-is — nothing is refit on test data:
- p99.9% caps → reused from `caps` dict computed on `app_train`
- Secondary table aggregations → shared, not recomputed
- Target encoding → applied at inference time using full-train mapping

**Pipeline steps:**
1. `run_cleaning` → `app_test_clean`
2. `run_main_features` → `app_test_features`
3. `run_final_join` → `app_test_joined` (LEFT JOIN with shared secondary tables)
4. `run_cross_table_features` → `app_test_cross`
5. `run_categorical_encoding` → `app_test_final`

**Output**: `app_test_final` — 48,744 rows, identical columns to
`app_train_final_v3` (excluding `TARGET`), verified by alignment check.

In [5]:
# ── Reusable pipeline functions ───────────────────────────────────────
# Each function accepts source/target table name parameters so the
# exact same logic runs on both app_train and app_test.
# Secondary table aggregations (bureau, installments etc.) are shared
# and do not need to be recomputed for the test set.

def run_cleaning(con, source: str, target: str, caps: dict):
    """
    Block 1 — Cleaning, generalised.
    Applies the same cleaning logic to any source table.

    caps: dict with precomputed p99.9% values from app_train —
    must be passed in, never recomputed from the source table
    to avoid leakage when source = app_test.
    """
    CAT_COLS_TO_CLEAN = [
        "NAME_TYPE_SUITE", "OCCUPATION_TYPE", "FONDKAPREMONT_MODE",
        "HOUSETYPE_MODE", "WALLSMATERIAL_MODE",
    ]
    xna_exprs = ", ".join([f"""
        CASE WHEN {col} IN ('XNA', 'XAP', 'Unknown')
             THEN NULL ELSE {col}
        END AS {col}
    """ for col in CAT_COLS_TO_CLEAN])
     
    emergencystate_expr = """
    CASE WHEN CAST(EMERGENCYSTATE_MODE AS VARCHAR) IN ('XNA', 'XAP', 'Unknown', 'true', 'false')
         THEN NULL
         ELSE CAST(EMERGENCYSTATE_MODE AS VARCHAR)
    END AS EMERGENCYSTATE_MODE
    """
    con.execute(f"""
        CREATE OR REPLACE TABLE {target} AS
        SELECT
            SK_ID_CURR,

            -- TARGET only exists in train — use NULL for test
            {'TARGET,' if source == 'app_train_features_tbl' else 'NULL AS TARGET,'}

            CASE WHEN DAYS_EMPLOYED = 365243 THEN NULL
                 ELSE ABS(DAYS_EMPLOYED) END             AS DAYS_EMPLOYED,
            CASE WHEN DAYS_EMPLOYED = 365243 THEN 1
                 ELSE 0 END                              AS IS_NOT_EMPLOYED,
            ABS(DAYS_BIRTH)                              AS DAYS_BIRTH,
            ABS(DAYS_REGISTRATION)                       AS DAYS_REGISTRATION,
            ABS(DAYS_ID_PUBLISH)                         AS DAYS_ID_PUBLISH,
            ABS(DAYS_LAST_PHONE_CHANGE)                  AS DAYS_LAST_PHONE_CHANGE,
            LEAST(AMT_INCOME_TOTAL, {caps['income_cap']})  AS AMT_INCOME_TOTAL,
            LEAST(AMT_CREDIT,       {caps['credit_cap']})  AS AMT_CREDIT,
            LEAST(AMT_ANNUITY,      {caps['annuity_cap']}) AS AMT_ANNUITY,
            LEAST(AMT_GOODS_PRICE,  {caps['goods_cap']})   AS AMT_GOODS_PRICE,
            CASE WHEN FLAG_OWN_CAR = 'N' THEN NULL
                 ELSE OWN_CAR_AGE END                    AS OWN_CAR_AGE,
            CASE WHEN APARTMENTS_AVG IS NULL THEN 0
                 ELSE 1 END                              AS HAS_APPRAISAL,
            {xna_exprs},
            {emergencystate_expr},
            CODE_GENDER, NAME_CONTRACT_TYPE, NAME_INCOME_TYPE,
            NAME_EDUCATION_TYPE, NAME_FAMILY_STATUS, NAME_HOUSING_TYPE,
            FLAG_OWN_CAR, FLAG_OWN_REALTY,
            CNT_CHILDREN, CNT_FAM_MEMBERS,
            EXT_SOURCE_1, EXT_SOURCE_2, EXT_SOURCE_3,
            REGION_RATING_CLIENT, REGION_RATING_CLIENT_W_CITY,
            REGION_POPULATION_RELATIVE,
            REG_CITY_NOT_WORK_CITY, REG_CITY_NOT_LIVE_CITY,
            REG_REGION_NOT_WORK_REGION, LIVE_CITY_NOT_WORK_CITY,
            LIVE_REGION_NOT_WORK_REGION,
            FLAG_MOBIL, FLAG_EMP_PHONE, FLAG_WORK_PHONE,
            FLAG_CONT_MOBILE, FLAG_PHONE, FLAG_EMAIL,
            HOUR_APPR_PROCESS_START,
            DEF_30_CNT_SOCIAL_CIRCLE, DEF_60_CNT_SOCIAL_CIRCLE,
            OBS_30_CNT_SOCIAL_CIRCLE, OBS_60_CNT_SOCIAL_CIRCLE,
            AMT_REQ_CREDIT_BUREAU_HOUR, AMT_REQ_CREDIT_BUREAU_DAY,
            AMT_REQ_CREDIT_BUREAU_WEEK, AMT_REQ_CREDIT_BUREAU_MON,
            AMT_REQ_CREDIT_BUREAU_QRT,  AMT_REQ_CREDIT_BUREAU_YEAR,
            ORGANIZATION_TYPE,
            FLAG_DOCUMENT_2,  FLAG_DOCUMENT_3,  FLAG_DOCUMENT_4,
            FLAG_DOCUMENT_5,  FLAG_DOCUMENT_6,  FLAG_DOCUMENT_7,
            FLAG_DOCUMENT_8,  FLAG_DOCUMENT_9,  FLAG_DOCUMENT_10,
            FLAG_DOCUMENT_11, FLAG_DOCUMENT_12, FLAG_DOCUMENT_13,
            FLAG_DOCUMENT_14, FLAG_DOCUMENT_15, FLAG_DOCUMENT_16,
            FLAG_DOCUMENT_17, FLAG_DOCUMENT_18, FLAG_DOCUMENT_19,
            FLAG_DOCUMENT_20, FLAG_DOCUMENT_21,
            APARTMENTS_AVG, APARTMENTS_MEDI, APARTMENTS_MODE,
            BASEMENTAREA_AVG, BASEMENTAREA_MEDI, BASEMENTAREA_MODE,
            YEARS_BEGINEXPLUATATION_AVG, YEARS_BEGINEXPLUATATION_MEDI,
            YEARS_BEGINEXPLUATATION_MODE,
            YEARS_BUILD_AVG, YEARS_BUILD_MEDI, YEARS_BUILD_MODE,
            COMMONAREA_AVG, COMMONAREA_MEDI, COMMONAREA_MODE,
            ELEVATORS_AVG, ELEVATORS_MEDI, ELEVATORS_MODE,
            ENTRANCES_AVG, ENTRANCES_MEDI, ENTRANCES_MODE,
            FLOORSMAX_AVG, FLOORSMAX_MEDI, FLOORSMAX_MODE,
            FLOORSMIN_AVG, FLOORSMIN_MEDI, FLOORSMIN_MODE,
            LANDAREA_AVG, LANDAREA_MEDI, LANDAREA_MODE,
            LIVINGAPARTMENTS_AVG, LIVINGAPARTMENTS_MEDI, LIVINGAPARTMENTS_MODE,
            LIVINGAREA_AVG, LIVINGAREA_MEDI, LIVINGAREA_MODE,
            NONLIVINGAPARTMENTS_AVG, NONLIVINGAPARTMENTS_MEDI,
            NONLIVINGAPARTMENTS_MODE,
            NONLIVINGAREA_AVG, NONLIVINGAREA_MEDI, NONLIVINGAREA_MODE,
            TOTALAREA_MODE
        FROM {source}
    """)
    print(f"  [{target}] cleaning done.")

In [6]:
def run_main_features(con, source: str, target: str):
    """
    Block 2 — Main table feature engineering, generalised.
    Reads from {source} (cleaned table), writes to {target}.
    No target-dependent computations — safe for test set.
    """
    con.execute(f"""
        CREATE OR REPLACE TABLE {target} AS
        SELECT
            SK_ID_CURR,
            TARGET,

            -- Financial ratios
            ROUND(AMT_ANNUITY   / NULLIF(AMT_INCOME_TOTAL, 0), 4) AS ANNUITY_INCOME_RATIO,
            ROUND(AMT_CREDIT    / NULLIF(AMT_INCOME_TOTAL, 0), 4) AS CREDIT_INCOME_RATIO,
            ROUND(AMT_CREDIT    / NULLIF(AMT_ANNUITY, 0),      2) AS CREDIT_TERM,
            ROUND(AMT_GOODS_PRICE / NULLIF(AMT_CREDIT, 0),     4) AS GOODS_CREDIT_RATIO,
            AMT_CREDIT - AMT_GOODS_PRICE                           AS CREDIT_GOODS_DIFF,
            ROUND(AMT_INCOME_TOTAL / NULLIF(CNT_FAM_MEMBERS, 0), 2) AS INCOME_PER_PERSON,
            ROUND(AMT_ANNUITY / NULLIF(CNT_FAM_MEMBERS, 0),    2) AS ANNUITY_PER_PERSON,
            ROUND(AMT_ANNUITY / NULLIF(AMT_GOODS_PRICE, 0),    4) AS ANNUITY_GOODS_RATIO,

            -- Temporal features
            ROUND(DAYS_BIRTH        / 365.25, 2) AS AGE_YEARS,
            ROUND(DAYS_REGISTRATION / 365.25, 2) AS REGISTRATION_YEARS,
            ROUND(DAYS_ID_PUBLISH   / 365.25, 2) AS ID_PUBLISH_YEARS,
            ROUND(DAYS_LAST_PHONE_CHANGE / 365.25, 2) AS PHONE_CHANGE_YEARS,
            ROUND(DAYS_EMPLOYED     / 365.25, 2) AS EMPLOYED_YEARS,
            IS_NOT_EMPLOYED,
            ROUND((DAYS_BIRTH - DAYS_EMPLOYED) / 365.25, 2) AS EMPLOYMENT_START_AGE,
            ROUND(DAYS_EMPLOYED / NULLIF(DAYS_BIRTH, 0), 4) AS EMPLOYED_TO_LIFE_RATIO,
            DAYS_ID_PUBLISH - DAYS_REGISTRATION              AS ID_REGISTRATION_GAP,
            ROUND((DAYS_ID_PUBLISH / 365.25) /
                  NULLIF(DAYS_BIRTH / 365.25, 0), 4)         AS ID_PUBLISH_TO_AGE,

            -- EXT_SOURCE combinations
            ROUND((COALESCE(EXT_SOURCE_1,0) + COALESCE(EXT_SOURCE_2,0) +
                   COALESCE(EXT_SOURCE_3,0)) /
                  NULLIF(
                      (CASE WHEN EXT_SOURCE_1 IS NOT NULL THEN 1 ELSE 0 END +
                       CASE WHEN EXT_SOURCE_2 IS NOT NULL THEN 1 ELSE 0 END +
                       CASE WHEN EXT_SOURCE_3 IS NOT NULL THEN 1 ELSE 0 END), 0
                  ), 4)                                       AS EXT_SOURCE_MEAN,
            ROUND(COALESCE(EXT_SOURCE_1,1) *
                  COALESCE(EXT_SOURCE_2,1) *
                  COALESCE(EXT_SOURCE_3,1), 6)               AS EXT_SOURCE_PROD,
            LEAST(COALESCE(EXT_SOURCE_1,1),
                  COALESCE(EXT_SOURCE_2,1),
                  COALESCE(EXT_SOURCE_3,1))                  AS EXT_SOURCE_MIN,
            GREATEST(COALESCE(EXT_SOURCE_1,0),
                     COALESCE(EXT_SOURCE_2,0),
                     COALESCE(EXT_SOURCE_3,0))               AS EXT_SOURCE_MAX,
            ROUND(SQRT((
                POWER(COALESCE(EXT_SOURCE_1,0)-(COALESCE(EXT_SOURCE_1,0)+COALESCE(EXT_SOURCE_2,0)+COALESCE(EXT_SOURCE_3,0))/3.0,2)+
                POWER(COALESCE(EXT_SOURCE_2,0)-(COALESCE(EXT_SOURCE_1,0)+COALESCE(EXT_SOURCE_2,0)+COALESCE(EXT_SOURCE_3,0))/3.0,2)+
                POWER(COALESCE(EXT_SOURCE_3,0)-(COALESCE(EXT_SOURCE_1,0)+COALESCE(EXT_SOURCE_2,0)+COALESCE(EXT_SOURCE_3,0))/3.0,2)
            )/3.0), 4)                                        AS EXT_SOURCE_STD,
            ROUND(EXT_SOURCE_1 - EXT_SOURCE_2, 4)            AS EXT_12_DIFF,
            ROUND(EXT_SOURCE_2 - EXT_SOURCE_3, 4)            AS EXT_23_DIFF,
            ROUND(EXT_SOURCE_2 * AMT_INCOME_TOTAL, 2)        AS EXT2_INCOME,
            ROUND(EXT_SOURCE_2 / NULLIF(
                AMT_CREDIT / NULLIF(AMT_INCOME_TOTAL, 0), 0), 4) AS EXT2_CREDIT_RATIO,
            ROUND(EXT_SOURCE_3 * (DAYS_BIRTH / 365.25), 4)   AS EXT3_AGE,
            (CASE WHEN EXT_SOURCE_1 IS NULL THEN 1 ELSE 0 END +
             CASE WHEN EXT_SOURCE_2 IS NULL THEN 1 ELSE 0 END +
             CASE WHEN EXT_SOURCE_3 IS NULL THEN 1 ELSE 0 END) AS EXT_MISSING_COUNT,
            EXT_SOURCE_1, EXT_SOURCE_2, EXT_SOURCE_3,

            -- Flag aggregations
            (COALESCE(FLAG_DOCUMENT_2,0)  + COALESCE(FLAG_DOCUMENT_3,0)  +
             COALESCE(FLAG_DOCUMENT_4,0)  + COALESCE(FLAG_DOCUMENT_5,0)  +
             COALESCE(FLAG_DOCUMENT_6,0)  + COALESCE(FLAG_DOCUMENT_7,0)  +
             COALESCE(FLAG_DOCUMENT_8,0)  + COALESCE(FLAG_DOCUMENT_9,0)  +
             COALESCE(FLAG_DOCUMENT_10,0) + COALESCE(FLAG_DOCUMENT_11,0) +
             COALESCE(FLAG_DOCUMENT_12,0) + COALESCE(FLAG_DOCUMENT_13,0) +
             COALESCE(FLAG_DOCUMENT_14,0) + COALESCE(FLAG_DOCUMENT_15,0) +
             COALESCE(FLAG_DOCUMENT_16,0) + COALESCE(FLAG_DOCUMENT_17,0) +
             COALESCE(FLAG_DOCUMENT_18,0) + COALESCE(FLAG_DOCUMENT_19,0) +
             COALESCE(FLAG_DOCUMENT_20,0) + COALESCE(FLAG_DOCUMENT_21,0)
            )                                                 AS TOTAL_DOCS_PROVIDED,
            CASE WHEN (COALESCE(FLAG_DOCUMENT_3,0) + COALESCE(FLAG_DOCUMENT_5,0) +
                       COALESCE(FLAG_DOCUMENT_6,0) + COALESCE(FLAG_DOCUMENT_8,0)) > 0
                 THEN 1 ELSE 0 END                            AS HAS_ANY_DOCUMENT,
            COALESCE(FLAG_MOBIL,0) + COALESCE(FLAG_EMP_PHONE,0) +
            COALESCE(FLAG_WORK_PHONE,0) + COALESCE(FLAG_CONT_MOBILE,0) +
            COALESCE(FLAG_PHONE,0) + COALESCE(FLAG_EMAIL,0)  AS CONTACT_SCORE,
            CASE WHEN REG_CITY_NOT_WORK_CITY    = 0
                  AND REG_REGION_NOT_WORK_REGION = 0
                 THEN 1 ELSE 0 END                            AS LIVE_WORK_SAME_REGION,
            CASE WHEN DEF_30_CNT_SOCIAL_CIRCLE > 0
                   OR DEF_60_CNT_SOCIAL_CIRCLE > 0
                 THEN 1 ELSE 0 END                            AS ANY_SOCIAL_CIRCLE_DEF,
            ROUND(DEF_60_CNT_SOCIAL_CIRCLE /
                  NULLIF(OBS_60_CNT_SOCIAL_CIRCLE, 0), 4)    AS SOCIAL_CIRCLE_DEF_RATIO,
            COALESCE(AMT_REQ_CREDIT_BUREAU_HOUR,0) +
            COALESCE(AMT_REQ_CREDIT_BUREAU_DAY,0)  +
            COALESCE(AMT_REQ_CREDIT_BUREAU_WEEK,0)            AS ENQUIRIES_SHORT_TERM,
            COALESCE(AMT_REQ_CREDIT_BUREAU_MON,0) +
            COALESCE(AMT_REQ_CREDIT_BUREAU_QRT,0) +
            COALESCE(AMT_REQ_CREDIT_BUREAU_YEAR,0)            AS ENQUIRIES_LONG_TERM,

            -- Categorical encoding
            CASE WHEN FLAG_OWN_CAR    = 'Y' THEN 1 ELSE 0 END AS HAS_CAR,
            CASE WHEN FLAG_OWN_REALTY = 'Y' THEN 1 ELSE 0 END AS HAS_REALTY,
            CASE WHEN CODE_GENDER     = 'M' THEN 1
                 WHEN CODE_GENDER     = 'F' THEN 0
                 ELSE NULL END                                 AS IS_MALE,
            CASE NAME_EDUCATION_TYPE
                WHEN 'Lower secondary'               THEN 1
                WHEN 'Secondary / secondary special' THEN 2
                WHEN 'Incomplete higher'             THEN 3
                WHEN 'Higher education'              THEN 4
                WHEN 'Academic degree'               THEN 5
                ELSE NULL END                                  AS EDUCATION_LEVEL,
            REGION_RATING_CLIENT,
            REGION_RATING_CLIENT_W_CITY,
            CASE WHEN NAME_CONTRACT_TYPE = 'Cash loans'         THEN 1 ELSE 0 END AS IS_CASH_LOAN,
            CASE WHEN NAME_INCOME_TYPE   = 'Working'            THEN 1 ELSE 0 END AS INCOME_WORKING,
            CASE WHEN NAME_INCOME_TYPE   = 'State servant'      THEN 1 ELSE 0 END AS INCOME_STATE_SERVANT,
            CASE WHEN NAME_INCOME_TYPE   = 'Pensioner'          THEN 1 ELSE 0 END AS INCOME_PENSIONER,
            CASE WHEN NAME_INCOME_TYPE   = 'Commercial associate' THEN 1 ELSE 0 END AS INCOME_COMMERCIAL,
            CASE WHEN NAME_HOUSING_TYPE  = 'House / apartment'  THEN 1 ELSE 0 END AS HOUSING_HOUSE,
            CASE WHEN NAME_HOUSING_TYPE  = 'With parents'       THEN 1 ELSE 0 END AS HOUSING_WITH_PARENTS,
            CASE WHEN NAME_HOUSING_TYPE  = 'Municipal apartment' THEN 1 ELSE 0 END AS HOUSING_MUNICIPAL,
            CASE WHEN NAME_FAMILY_STATUS = 'Married'            THEN 1 ELSE 0 END AS IS_MARRIED,
            CASE WHEN NAME_FAMILY_STATUS = 'Single / not married' THEN 1 ELSE 0 END AS IS_SINGLE,
            CASE WHEN FLAG_OWN_CAR = 'Y'
                  AND FLAG_OWN_REALTY = 'N' THEN 1 ELSE 0 END AS HAS_CAR_NOT_REALTY,
            CASE WHEN AMT_ANNUITY / NULLIF(AMT_INCOME_TOTAL,0) > 0.35
                 THEN 1 ELSE 0 END                             AS HIGH_ANNUITY_BURDEN,
            CASE WHEN AMT_CREDIT > AMT_GOODS_PRICE
                 THEN 1 ELSE 0 END                             AS CREDIT_EXCEEDS_GOODS,

            -- Pass through for target encoding at model time
            OCCUPATION_TYPE,
            ORGANIZATION_TYPE,

            -- Raw numerics
            AMT_INCOME_TOTAL, AMT_CREDIT, AMT_ANNUITY, AMT_GOODS_PRICE,
            DAYS_BIRTH, DAYS_REGISTRATION, DAYS_ID_PUBLISH,
            DAYS_LAST_PHONE_CHANGE, DAYS_EMPLOYED,
            CNT_CHILDREN, CNT_FAM_MEMBERS, HOUR_APPR_PROCESS_START,
            DEF_30_CNT_SOCIAL_CIRCLE, DEF_60_CNT_SOCIAL_CIRCLE,
            OBS_30_CNT_SOCIAL_CIRCLE, OBS_60_CNT_SOCIAL_CIRCLE,
            REGION_POPULATION_RELATIVE,
            REG_CITY_NOT_WORK_CITY, REG_CITY_NOT_LIVE_CITY,
            REG_REGION_NOT_WORK_REGION, LIVE_CITY_NOT_WORK_CITY,
            LIVE_REGION_NOT_WORK_REGION,
            FLAG_EMP_PHONE, FLAG_WORK_PHONE, FLAG_CONT_MOBILE,
            FLAG_PHONE, FLAG_EMAIL,
            FLAG_DOCUMENT_3, FLAG_DOCUMENT_5, FLAG_DOCUMENT_6,
            FLAG_DOCUMENT_8, FLAG_DOCUMENT_16, FLAG_DOCUMENT_18,
            HAS_APPRAISAL

        FROM {source}
    """)
    print(f"  [{target}] main features done.")

In [7]:
def run_cross_table_features(con, source: str, target: str):
    """
    Block 4 — Cross-table features, generalised.
    Reads from {source} (post-join table), writes to {target}.
    """
    con.execute(f"""
        CREATE OR REPLACE TABLE {target} AS
        SELECT
            *,
            ROUND(AMT_CREDIT / NULLIF(PREV_AMT_CREDIT_MEAN, 0), 4)
                                                AS CREDIT_VS_PREV_RATIO,
            ROUND(AMT_ANNUITY / NULLIF(PREV_AMT_ANNUITY_MEAN, 0), 4)
                                                AS ANNUITY_VS_PREV_RATIO,
            ROUND(AMT_CREDIT / NULLIF(PREV_AMT_APPLICATION_MEAN, 0), 4)
                                                AS CREDIT_VS_PREV_APPLICATION_RATIO,
            AMT_CREDIT + COALESCE(BUREAU_DEBT_SUM, 0)
                                                AS TOTAL_DEBT,
            ROUND(AMT_INCOME_TOTAL /
                  NULLIF(AMT_CREDIT + COALESCE(BUREAU_DEBT_SUM, 0), 0), 4)
                                                AS INCOME_TO_TOTAL_DEBT_RATIO,
            ROUND(AMT_ANNUITY /
                  NULLIF(AMT_INCOME_TOTAL - COALESCE(BUREAU_DEBT_MEAN, 0), 0), 4)
                                                AS ANNUITY_TO_RESIDUAL_INCOME,
            ROUND(COALESCE(BUREAU_OVERDUE_MEAN,0) /
                  NULLIF(AMT_INCOME_TOTAL, 0), 6) AS OVERDUE_TO_INCOME_RATIO,
            ROUND(COALESCE(BUREAU_OVERDUE_SUM,0) /
                  NULLIF(AMT_CREDIT, 0), 4)     AS OVERDUE_TO_CREDIT_RATIO,
            (COALESCE(POS_DPD_MAX,0) +
             COALESCE(INSTAL_DELAY_MAX,0) +
             COALESCE(CC_DPD_MAX,0))            AS TOTAL_DPD_SCORE,
            CASE WHEN COALESCE(POS_EVER_LATE,0)   = 1
                   OR COALESCE(INSTAL_EVER_LATE,0) = 1
                   OR COALESCE(CC_EVER_LATE,0)     = 1
                 THEN 1 ELSE 0 END              AS EVER_LATE_ANY_PRODUCT,
            CASE WHEN COALESCE(POS_EVER_LATE,0)   = 1
                  AND COALESCE(INSTAL_EVER_LATE,0) = 1
                  AND COALESCE(CC_EVER_LATE,0)     = 1
                 THEN 1 ELSE 0 END              AS EVER_LATE_ALL_PRODUCTS,
            (COALESCE(POS_EVER_LATE,0)   +
             COALESCE(INSTAL_EVER_LATE,0) +
             COALESCE(CC_EVER_LATE,0)    +
             COALESCE(BBAL_EVER_LATE,0))        AS N_PRODUCTS_EVER_LATE,
            COALESCE(PREV_APP_COUNT,0)     +
            COALESCE(POS_CONTRACT_COUNT,0) +
            COALESCE(CC_CONTRACT_COUNT,0)       AS TOTAL_HC_CONTRACTS,
            COALESCE(BBAL_MONTHS_COUNT,0)  +
            COALESCE(POS_MONTHS_COUNT,0)   +
            COALESCE(INSTAL_COUNT,0)       +
            COALESCE(CC_MONTHS_COUNT,0)         AS TOTAL_HISTORY_MONTHS,
            ROUND(EXT_SOURCE_MEAN *
                  (1 - LEAST(COALESCE(BBAL_DPD_RATIO,0), 1)), 4)
                                                AS EXT_SCORE_VS_INTERNAL_DPD,
            ROUND(EXT_SOURCE_MEAN *
                  (1 - COALESCE(PREV_REFUSAL_RATE,0)), 4)
                                                AS EXT_SCORE_ADJ_REFUSAL,
            ROUND((COALESCE(INSTAL_LATE_RATIO,0) +
                   COALESCE(POS_DPD_MEAN,0) / 30.0) / 2.0, 4)
                                                AS AVG_LATE_RATIO_HC,
            ROUND(COALESCE(INSTAL_DELAY_TREND,0) +
                  COALESCE(POS_DPD_TREND,0)     +
                  COALESCE(CC_UTILIZATION_TREND,0), 4)
                                                AS COMBINED_WORSENING_TREND,
            ROUND(COALESCE(CC_UTILIZATION_MEAN,0) +
                  COALESCE(ANNUITY_INCOME_RATIO,0), 4)
                                                AS FINANCIAL_STRESS_SCORE,
            CASE WHEN COALESCE(CC_EVER_OVER_LIMIT,0) = 1
                  AND COALESCE(INSTAL_EVER_LATE,0)   = 1
                 THEN 1 ELSE 0 END              AS OVER_LIMIT_AND_LATE
        FROM {source}
    """)
    print(f"  [{target}] cross-table features done.")


def run_categorical_encoding(con, source: str, target: str):
    """
    Block 5 — One-hot encode PREV_LAST_STATUS.
    OCCUPATION_TYPE and ORGANIZATION_TYPE are kept as VARCHAR
    for target encoding inside the k-fold loop.
    """
    con.execute(f"""
        CREATE OR REPLACE TABLE {target} AS
        SELECT
            *,
            CASE WHEN PREV_LAST_STATUS = 'Approved' THEN 1 ELSE 0 END
                                                AS PREV_LAST_APPROVED,
            CASE WHEN PREV_LAST_STATUS = 'Refused'  THEN 1 ELSE 0 END
                                                AS PREV_LAST_REFUSED,
            CASE WHEN PREV_LAST_STATUS = 'Canceled' THEN 1 ELSE 0 END
                                                AS PREV_LAST_CANCELED
        FROM {source}
    """)
    print(f"  [{target}] categorical encoding done.")

In [8]:
def run_final_join(con, source_features: str, target: str):
    """
    Block 3g — Final join, generalised.
    Joins the main feature table with all secondary aggregations.
    Secondary tables are shared between train and test — no recomputation needed.
    Split into three steps to avoid memory errors.
    """

    print("Step 1/3 — bureau features...")
    con.execute(f"""
        CREATE OR REPLACE TABLE _join_temp1 AS
        SELECT a.*,
            b.BUREAU_LOAN_COUNT, b.BUREAU_ACTIVE_COUNT, b.BUREAU_CLOSED_COUNT,
            b.BUREAU_ACTIVE_RATIO, b.BUREAU_CREDIT_SUM, b.BUREAU_CREDIT_MEAN,
            b.BUREAU_CREDIT_MAX, b.BUREAU_DEBT_SUM, b.BUREAU_DEBT_MEAN,
            b.BUREAU_OVERDUE_SUM, b.BUREAU_OVERDUE_MEAN, b.BUREAU_MAX_DAY_OVERDUE,
            b.BUREAU_MEAN_DAY_OVERDUE, b.BUREAU_OVERDUE_COUNT,
            b.BUREAU_DEBT_CREDIT_RATIO, b.BUREAU_DAYS_CREDIT_MIN,
            b.BUREAU_DAYS_CREDIT_MEAN, b.BUREAU_ENDDATE_MAX,
            b.BUREAU_DAYS_UPDATE_MEAN, b.BUREAU_PROLONG_SUM, b.BUREAU_PROLONG_MEAN,
            b.BUREAU_CONSUMER_COUNT, b.BUREAU_CARD_COUNT,
            b.BUREAU_MORTGAGE_COUNT, b.BUREAU_CAR_COUNT,
            b.BUREAU_HAS_BAD_DEBT, b.BUREAU_EVER_OVERDUE,
            bb.BBAL_MONTHS_COUNT, bb.BBAL_UNIQUE_MONTHS, bb.BBAL_STATUS_C_RATIO,
            bb.BBAL_STATUS_0_RATIO, bb.BBAL_DPD_RATIO, bb.BBAL_DPD_MAX,
            bb.BBAL_DPD_MEAN, bb.BBAL_DPD_COUNT, bb.BBAL_EVER_LATE,
            bb.BBAL_EVER_LATE_3M, bb.BBAL_DPD_MEAN_12M, bb.BBAL_DPD_MAX_12M,
            bb.BBAL_DPD_MEAN_6M, bb.BBAL_DPD_TREND
        FROM {source_features}        a
        LEFT JOIN bureau_features     b  USING (SK_ID_CURR)
        LEFT JOIN bureau_bal_features bb USING (SK_ID_CURR)
    """)
    print("Step 1/3 — done.")

    print("Step 2/3 — prev_app and pos features...")
    con.execute(f"""
        CREATE OR REPLACE TABLE _join_temp2 AS
        SELECT t.*,
            p.PREV_APP_COUNT, p.PREV_APPROVED_COUNT, p.PREV_REFUSED_COUNT,
            p.PREV_CANCELED_COUNT, p.PREV_APPROVAL_RATE, p.PREV_REFUSAL_RATE,
            p.PREV_AMT_CREDIT_MEAN, p.PREV_AMT_CREDIT_MAX,
            p.PREV_AMT_APPLICATION_MEAN, p.PREV_CREDIT_APPLICATION_RATIO,
            p.PREV_AMT_ANNUITY_MEAN, p.PREV_AMT_ANNUITY_MAX,
            p.PREV_DOWN_PAYMENT_MEAN, p.PREV_DOWN_PAYMENT_RATE_MEAN,
            p.PREV_CNT_PAYMENT_MEAN, p.PREV_CNT_PAYMENT_MAX,
            p.PREV_DAYS_DECISION_LAST, p.PREV_DAYS_DECISION_MEAN,
            p.PREV_CONSUMER_COUNT, p.PREV_CASH_COUNT, p.PREV_REVOLVING_COUNT,
            p.PREV_LAST_STATUS, p.PREV_LAST_CREDIT, p.PREV_LAST_CNT_PAYMENT,
            p.PREV_EVER_REFUSED, p.PREV_EVER_HIGH_YIELD,
            p.PREV_NFLAG_INSURED_MEAN, p.PREV_INTEREST_RATE_MEAN,
            p.PREV_AMT_CREDIT_MEAN_12M, p.PREV_APP_COUNT_12M,
            p.PREV_APPROVAL_RATE_12M, p.PREV_CASH_CREDIT_MEAN,
            p.PREV_REVOLVING_CREDIT_MEAN, p.PREV_CONSUMER_CREDIT_MEAN,
            p.PREV_CASH_ANNUITY_MEAN, p.PREV_REVOLVING_ANNUITY_MEAN,
            pos.POS_MONTHS_COUNT, pos.POS_CONTRACT_COUNT, pos.POS_COMPLETED_RATIO,
            pos.POS_EVER_COMPLETED, pos.POS_ACTIVE_COUNT,
            pos.POS_DPD_MAX, pos.POS_DPD_MEAN, pos.POS_DPD_DEF_MAX,
            pos.POS_DPD_DEF_MEAN, pos.POS_DPD_COUNT, pos.POS_EVER_LATE,
            pos.POS_EVER_LATE_30, pos.POS_EVER_LATE_60,
            pos.POS_DPD_MEAN_12M, pos.POS_DPD_MAX_12M, pos.POS_DPD_MEAN_6M,
            pos.POS_DPD_COUNT_12M, pos.POS_DPD_TREND,
            pos.POS_CNT_INSTALMENT_MEAN, pos.POS_CNT_INSTALMENT_FUTURE_MEAN
        FROM _join_temp1            t
        LEFT JOIN prev_app_features p   USING (SK_ID_CURR)
        LEFT JOIN pos_features      pos USING (SK_ID_CURR)
    """)
    con.execute("DROP TABLE IF EXISTS _join_temp1")
    print("Step 2/3 — done.")

    print("Step 3/3 — installments and cc features...")
    con.execute(f"""
        CREATE OR REPLACE TABLE {target} AS
        SELECT t.*,
            i.INSTAL_COUNT, i.INSTAL_CONTRACT_COUNT, i.INSTAL_LATE_COUNT,
            i.INSTAL_LATE_RATIO, i.INSTAL_EVER_LATE, i.INSTAL_EVER_LATE_30,
            i.INSTAL_EARLY_COUNT, i.INSTAL_DELAY_MEAN, i.INSTAL_DELAY_MAX,
            i.INSTAL_DELAY_MEAN_LATE, i.INSTAL_EARLY_DAYS_MEAN,
            i.INSTAL_PAYMENT_RATIO_MEAN, i.INSTAL_PAYMENT_RATIO_MIN,
            i.INSTAL_UNDERPAID_COUNT, i.INSTAL_EVER_UNDERPAID,
            i.INSTAL_AMT_MEAN, i.INSTAL_PAYMENT_MEAN, i.INSTAL_AMT_DIFF_SUM,
            i.INSTAL_DELAY_MEAN_12M, i.INSTAL_DELAY_MAX_12M,
            i.INSTAL_DELAY_MEAN_6M, i.INSTAL_LATE_RATIO_12M,
            i.INSTAL_DELAY_TREND,
            cc.CC_MONTHS_COUNT, cc.CC_CONTRACT_COUNT,
            cc.CC_BALANCE_MEAN, cc.CC_BALANCE_MAX,
            cc.CC_LIMIT_MEAN, cc.CC_LIMIT_MAX,
            cc.CC_UTILIZATION_MEAN, cc.CC_UTILIZATION_MAX,
            cc.CC_OVER_LIMIT_COUNT, cc.CC_EVER_OVER_LIMIT,
            cc.CC_PAYMENT_RATIO_MEAN, cc.CC_PAYMENT_RATIO_MIN,
            cc.CC_PAYMENT_MEAN, cc.CC_DRAWINGS_MEAN,
            cc.CC_ATM_DRAWINGS_MEAN, cc.CC_ATM_RATIO_MEAN,
            cc.CC_CNT_DRAWINGS_MEAN, cc.CC_CNT_ATM_DRAWINGS_MEAN,
            cc.CC_RECEIVABLE_MEAN, cc.CC_RECEIVABLE_MAX,
            cc.CC_DPD_MAX, cc.CC_DPD_MEAN, cc.CC_DPD_DEF_MAX,
            cc.CC_DPD_COUNT, cc.CC_EVER_LATE,
            cc.CC_UTILIZATION_MEAN_12M, cc.CC_DPD_MEAN_12M,
            cc.CC_DPD_MAX_12M, cc.CC_UTILIZATION_TREND
        FROM _join_temp2              t
        LEFT JOIN installments_features i  USING (SK_ID_CURR)
        LEFT JOIN cc_features           cc USING (SK_ID_CURR)
    """)
    con.execute("DROP TABLE IF EXISTS _join_temp2")
    print("Step 3/3 — done.")

In [9]:
# Same query as inside build_clean_view, now saved to a variable

caps_df = con.execute("""
    SELECT
        PERCENTILE_CONT(0.999) WITHIN GROUP
            (ORDER BY AMT_INCOME_TOTAL)    AS income_cap,
        PERCENTILE_CONT(0.999) WITHIN GROUP
            (ORDER BY AMT_CREDIT)          AS credit_cap,
        PERCENTILE_CONT(0.999) WITHIN GROUP
            (ORDER BY AMT_ANNUITY)         AS annuity_cap,
        PERCENTILE_CONT(0.999) WITHIN GROUP
            (ORDER BY AMT_GOODS_PRICE)     AS goods_cap
    FROM app_train
""").fetchdf().iloc[0]

caps = {
    'income_cap':  caps_df['income_cap'],
    'credit_cap':  caps_df['credit_cap'],
    'annuity_cap': caps_df['annuity_cap'],
    'goods_cap':   caps_df['goods_cap'],
}

print("Caps loaded from app_train:")
for k, v in caps.items():
    print(f"  {k:<15}: {v:>15,.0f}")

Caps loaded from app_train:
  income_cap     :         900,000
  credit_cap     :       2,517,300
  annuity_cap    :         110,048
  goods_cap      :       2,250,000


In [10]:
print("Block 1 — cleaning app_test...")
run_cleaning(con,
    source='app_test',
    target='app_test_clean',
    caps=caps)

Block 1 — cleaning app_test...
  [app_test_clean] cleaning done.


In [11]:
print("Block 2 — main features...")
run_main_features(con,
    source='app_test_clean',
    target='app_test_features')

Block 2 — main features...
  [app_test_features] main features done.


In [12]:
print("Block 3 step 1/3 — bureau join...")
con.execute("""
    CREATE OR REPLACE TABLE _join_temp1 AS
    SELECT a.*,
        b.BUREAU_LOAN_COUNT, b.BUREAU_ACTIVE_COUNT, b.BUREAU_CLOSED_COUNT,
        b.BUREAU_ACTIVE_RATIO, b.BUREAU_CREDIT_SUM, b.BUREAU_CREDIT_MEAN,
        b.BUREAU_CREDIT_MAX, b.BUREAU_DEBT_SUM, b.BUREAU_DEBT_MEAN,
        b.BUREAU_OVERDUE_SUM, b.BUREAU_OVERDUE_MEAN, b.BUREAU_MAX_DAY_OVERDUE,
        b.BUREAU_MEAN_DAY_OVERDUE, b.BUREAU_OVERDUE_COUNT,
        b.BUREAU_DEBT_CREDIT_RATIO, b.BUREAU_DAYS_CREDIT_MIN,
        b.BUREAU_DAYS_CREDIT_MEAN, b.BUREAU_ENDDATE_MAX,
        b.BUREAU_DAYS_UPDATE_MEAN, b.BUREAU_PROLONG_SUM, b.BUREAU_PROLONG_MEAN,
        b.BUREAU_CONSUMER_COUNT, b.BUREAU_CARD_COUNT,
        b.BUREAU_MORTGAGE_COUNT, b.BUREAU_CAR_COUNT,
        b.BUREAU_HAS_BAD_DEBT, b.BUREAU_EVER_OVERDUE,
        bb.BBAL_MONTHS_COUNT, bb.BBAL_UNIQUE_MONTHS, bb.BBAL_STATUS_C_RATIO,
        bb.BBAL_STATUS_0_RATIO, bb.BBAL_DPD_RATIO, bb.BBAL_DPD_MAX,
        bb.BBAL_DPD_MEAN, bb.BBAL_DPD_COUNT, bb.BBAL_EVER_LATE,
        bb.BBAL_EVER_LATE_3M, bb.BBAL_DPD_MEAN_12M, bb.BBAL_DPD_MAX_12M,
        bb.BBAL_DPD_MEAN_6M, bb.BBAL_DPD_TREND
    FROM app_test_features         a
    LEFT JOIN bureau_features      b  USING (SK_ID_CURR)
    LEFT JOIN bureau_bal_features  bb USING (SK_ID_CURR)
""")
print("done.")

Block 3 step 1/3 — bureau join...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

done.


In [13]:
print("Block 3 step 2/3 — prev_app and pos join...")
con.execute("""
    CREATE OR REPLACE TABLE _join_temp2 AS
    SELECT t.*,
        p.PREV_APP_COUNT, p.PREV_APPROVED_COUNT, p.PREV_REFUSED_COUNT,
        p.PREV_CANCELED_COUNT, p.PREV_APPROVAL_RATE, p.PREV_REFUSAL_RATE,
        p.PREV_AMT_CREDIT_MEAN, p.PREV_AMT_CREDIT_MAX,
        p.PREV_AMT_APPLICATION_MEAN, p.PREV_CREDIT_APPLICATION_RATIO,
        p.PREV_AMT_ANNUITY_MEAN, p.PREV_AMT_ANNUITY_MAX,
        p.PREV_DOWN_PAYMENT_MEAN, p.PREV_DOWN_PAYMENT_RATE_MEAN,
        p.PREV_CNT_PAYMENT_MEAN, p.PREV_CNT_PAYMENT_MAX,
        p.PREV_DAYS_DECISION_LAST, p.PREV_DAYS_DECISION_MEAN,
        p.PREV_CONSUMER_COUNT, p.PREV_CASH_COUNT, p.PREV_REVOLVING_COUNT,
        p.PREV_LAST_STATUS, p.PREV_LAST_CREDIT, p.PREV_LAST_CNT_PAYMENT,
        p.PREV_EVER_REFUSED, p.PREV_EVER_HIGH_YIELD,
        p.PREV_NFLAG_INSURED_MEAN, p.PREV_INTEREST_RATE_MEAN,
        p.PREV_AMT_CREDIT_MEAN_12M, p.PREV_APP_COUNT_12M,
        p.PREV_APPROVAL_RATE_12M, p.PREV_CASH_CREDIT_MEAN,
        p.PREV_REVOLVING_CREDIT_MEAN, p.PREV_CONSUMER_CREDIT_MEAN,
        p.PREV_CASH_ANNUITY_MEAN, p.PREV_REVOLVING_ANNUITY_MEAN,
        pos.POS_MONTHS_COUNT, pos.POS_CONTRACT_COUNT, pos.POS_COMPLETED_RATIO,
        pos.POS_EVER_COMPLETED, pos.POS_ACTIVE_COUNT,
        pos.POS_DPD_MAX, pos.POS_DPD_MEAN, pos.POS_DPD_DEF_MAX,
        pos.POS_DPD_DEF_MEAN, pos.POS_DPD_COUNT, pos.POS_EVER_LATE,
        pos.POS_EVER_LATE_30, pos.POS_EVER_LATE_60,
        pos.POS_DPD_MEAN_12M, pos.POS_DPD_MAX_12M, pos.POS_DPD_MEAN_6M,
        pos.POS_DPD_COUNT_12M, pos.POS_DPD_TREND,
        pos.POS_CNT_INSTALMENT_MEAN, pos.POS_CNT_INSTALMENT_FUTURE_MEAN
    FROM _join_temp1            t
    LEFT JOIN prev_app_features p   USING (SK_ID_CURR)
    LEFT JOIN pos_features      pos USING (SK_ID_CURR)
""")
con.execute("DROP TABLE IF EXISTS _join_temp1")
print("done.")

Block 3 step 2/3 — prev_app and pos join...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

done.


In [14]:
print("Block 3 step 3/3 — installments and cc join...")
con.execute("""
    CREATE OR REPLACE TABLE app_test_joined AS
    SELECT t.*,
        i.INSTAL_COUNT, i.INSTAL_CONTRACT_COUNT, i.INSTAL_LATE_COUNT,
        i.INSTAL_LATE_RATIO, i.INSTAL_EVER_LATE, i.INSTAL_EVER_LATE_30,
        i.INSTAL_EARLY_COUNT, i.INSTAL_DELAY_MEAN, i.INSTAL_DELAY_MAX,
        i.INSTAL_DELAY_MEAN_LATE, i.INSTAL_EARLY_DAYS_MEAN,
        i.INSTAL_PAYMENT_RATIO_MEAN, i.INSTAL_PAYMENT_RATIO_MIN,
        i.INSTAL_UNDERPAID_COUNT, i.INSTAL_EVER_UNDERPAID,
        i.INSTAL_AMT_MEAN, i.INSTAL_PAYMENT_MEAN, i.INSTAL_AMT_DIFF_SUM,
        i.INSTAL_DELAY_MEAN_12M, i.INSTAL_DELAY_MAX_12M,
        i.INSTAL_DELAY_MEAN_6M, i.INSTAL_LATE_RATIO_12M,
        i.INSTAL_DELAY_TREND,
        cc.CC_MONTHS_COUNT, cc.CC_CONTRACT_COUNT,
        cc.CC_BALANCE_MEAN, cc.CC_BALANCE_MAX,
        cc.CC_LIMIT_MEAN, cc.CC_LIMIT_MAX,
        cc.CC_UTILIZATION_MEAN, cc.CC_UTILIZATION_MAX,
        cc.CC_OVER_LIMIT_COUNT, cc.CC_EVER_OVER_LIMIT,
        cc.CC_PAYMENT_RATIO_MEAN, cc.CC_PAYMENT_RATIO_MIN,
        cc.CC_PAYMENT_MEAN, cc.CC_DRAWINGS_MEAN,
        cc.CC_ATM_DRAWINGS_MEAN, cc.CC_ATM_RATIO_MEAN,
        cc.CC_CNT_DRAWINGS_MEAN, cc.CC_CNT_ATM_DRAWINGS_MEAN,
        cc.CC_RECEIVABLE_MEAN, cc.CC_RECEIVABLE_MAX,
        cc.CC_DPD_MAX, cc.CC_DPD_MEAN, cc.CC_DPD_DEF_MAX,
        cc.CC_DPD_COUNT, cc.CC_EVER_LATE,
        cc.CC_UTILIZATION_MEAN_12M, cc.CC_DPD_MEAN_12M,
        cc.CC_DPD_MAX_12M, cc.CC_UTILIZATION_TREND
    FROM _join_temp2              t
    LEFT JOIN installments_features i  USING (SK_ID_CURR)
    LEFT JOIN cc_features           cc USING (SK_ID_CURR)
""")
con.execute("DROP TABLE IF EXISTS _join_temp2")
print("done.")

Block 3 step 3/3 — installments and cc join...
done.


In [15]:
print("Block 4 — cross-table features...")
run_cross_table_features(con,
    source='app_test_joined',
    target='app_test_cross')

Block 4 — cross-table features...
  [app_test_cross] cross-table features done.


In [16]:
print("Block 5 — categorical encoding...")
run_categorical_encoding(con,
    source='app_test_cross',
    target='app_test_final')

Block 5 — categorical encoding...
  [app_test_final] categorical encoding done.


In [17]:
# Cleanup intermediate tables
for t in ['app_test_clean', 'app_test_features',
          'app_test_joined', 'app_test_cross']:
    con.execute(f"DROP TABLE IF EXISTS {t}")
    print(f"Dropped: {t}")

print("\napp_test_final is ready.")

Dropped: app_test_clean
Dropped: app_test_features
Dropped: app_test_joined
Dropped: app_test_cross

app_test_final is ready.


In [21]:
# Patch — add HAS_APPRAISAL to app_train_final_v3 from app_train_clean
con.execute("""
    ALTER TABLE app_train_final_v3
    ADD COLUMN HAS_APPRAISAL INTEGER
""")

con.execute("""
    UPDATE app_train_final_v3 t
    SET HAS_APPRAISAL = c.HAS_APPRAISAL
    FROM app_train_clean c
    WHERE t.SK_ID_CURR = c.SK_ID_CURR
""")

# Verify
result = con.execute("""
    SELECT
        COUNT(*)                          AS total_rows,
        SUM(HAS_APPRAISAL)                AS has_appraisal_count,
        SUM(1 - HAS_APPRAISAL)            AS no_appraisal_count
    FROM app_train_final_v3
""").fetchdf()
print(result)

   total_rows  has_appraisal_count  no_appraisal_count
0      307511             151450.0            156061.0


In [22]:
# ── Final alignment check ─────────────────────────────────────────────
# Verify train and test have identical columns (excluding TARGET)

train_cols = set(con.execute("""
    SELECT column_name FROM information_schema.columns
    WHERE table_name = 'app_train_final_v3'
      AND column_name != 'TARGET'
    ORDER BY column_name
""").fetchdf()['column_name'])

test_cols = set(con.execute("""
    SELECT column_name FROM information_schema.columns
    WHERE table_name = 'app_test_final'
    ORDER BY column_name
""").fetchdf()['column_name'])

in_train_not_test = train_cols - test_cols
in_test_not_train = test_cols - train_cols

print(f"Train columns : {len(train_cols)}")
print(f"Test columns  : {len(test_cols)}")
print(f"In train not test : {in_train_not_test if in_train_not_test else 'None ✓'}")
print(f"In test not train : {in_test_not_train if in_test_not_train else 'None ✓'}")

Train columns : 268
Test columns  : 269
In train not test : None ✓
In test not train : {'TARGET'}


In [24]:
# Check if HAS_APPRAISAL is in each intermediate train table
for table in ['app_train_clean', 'app_train_features_tbl',
              'app_train_final', 'app_train_final_v2',
              'app_train_final_v3']:
    result = con.execute(f"""
        SELECT COUNT(*) FROM information_schema.columns
        WHERE table_name = '{table}'
          AND column_name = 'HAS_APPRAISAL'
    """).fetchone()[0]
    print(f"  {table:<35} HAS_APPRAISAL present: {result == 1}")

  app_train_clean                     HAS_APPRAISAL present: True
  app_train_features_tbl              HAS_APPRAISAL present: False
  app_train_final                     HAS_APPRAISAL present: False
  app_train_final_v2                  HAS_APPRAISAL present: False
  app_train_final_v3                  HAS_APPRAISAL present: True
